In [1]:
# ============================================================
# CELL 1 — INSTALL REQUIRED PACKAGES
# ============================================================

!pip install -q \
    streamlit \
    fastapi \
    uvicorn \
    pyngrok \
    requests \
    joblib \
    plotly \
    scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 69.2 MB/s eta 0:00:00


In [2]:
# ============================================================
# CELL 2 — IMPORT LIBRARIES
# ============================================================

import os
import glob
import json
import shutil
import subprocess
import time
import warnings

import numpy as np
import pandas as pd
import joblib

from IPython.display import display

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
# ============================================================
# CELL 3 — PROJECT CONFIGURATION
# ============================================================

PROJECT_ROOT = "/content/ANLOK-WATER-PREDICTION-PROJECT-"

GITHUB_REPO_URL = (
    "https://github.com/LulutheBA/"
    "ANLOK-WATER-PREDICTION-PROJECT-.git"
)

MODELLING_DATA_DIR = os.path.join(
    PROJECT_ROOT,
    "Outputs",
    "ModellingData"
)

MODELS_DIR = os.path.join(
    PROJECT_ROOT,
    "Outputs",
    "Models"
)

MODEL_EVALUATION_DIR = os.path.join(
    PROJECT_ROOT,
    "Outputs",
    "ModelEvaluation"
)

MODEL_VISUALISATION_DIR = os.path.join(
    PROJECT_ROOT,
    "Outputs",
    "ModelVisualisations"
)

MODEL_REPORTS_DIR = os.path.join(
    PROJECT_ROOT,
    "Outputs",
    "ModelReports"
)

print("Project configuration created.")

Project configuration created.


In [4]:
# ============================================================
# CELL 4 — CONNECT TO GITHUB
# ============================================================

if not os.path.isdir(os.path.join(PROJECT_ROOT, ".git")):

    print("Repository not found locally.")
    print("Cloning repository...")

    if os.path.exists(PROJECT_ROOT):
        shutil.rmtree(PROJECT_ROOT)

    clone_result = subprocess.run(
        [
            "git",
            "clone",
            GITHUB_REPO_URL,
            PROJECT_ROOT
        ],
        capture_output=True,
        text=True
    )

    if clone_result.returncode != 0:
        raise RuntimeError(
            "GitHub clone failed:\n"
            + clone_result.stderr
        )

    print("Repository cloned successfully.")

else:

    print("Existing repository detected.")
    print("Pulling latest changes...")

    fetch_result = subprocess.run(
        ["git", "fetch"],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True
    )

    pull_result = subprocess.run(
        ["git", "pull", "--ff-only"],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True
    )

    if pull_result.returncode != 0:
        print(pull_result.stderr)
    else:
        print("Repository updated successfully.")


os.chdir(PROJECT_ROOT)

# ------------------------------------------------------------
# Repository information
# ------------------------------------------------------------

branch = subprocess.run(
    ["git", "branch", "--show-current"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True
).stdout.strip()

latest_commit = subprocess.run(
    ["git", "log", "-1", "--oneline"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True
).stdout.strip()

print()
print("=" * 80)
print("GITHUB CONNECTION")
print("=" * 80)

print("Project root:", PROJECT_ROOT)
print("Branch:", branch)
print("Latest commit:", latest_commit)

Repository not found locally.
Cloning repository...
Repository cloned successfully.

GITHUB CONNECTION
Project root: /content/ANLOK-WATER-PREDICTION-PROJECT-
Branch: main
Latest commit: 540a2cb Add files via upload


In [5]:
# ============================================================
# CELL 5 — LOCATE NOTEBOOK 4
# ============================================================

nb4_candidates = []

patterns = [
    "*NB4*.ipynb",
    "*Notebook*4*.ipynb",
    "*04*Model*.ipynb"
]

for pattern in patterns:

    matches = glob.glob(
        os.path.join(
            PROJECT_ROOT,
            "**",
            pattern
        ),
        recursive=True
    )

    nb4_candidates.extend(matches)


# Remove duplicates
nb4_candidates = sorted(set(nb4_candidates))


print("=" * 80)
print("NOTEBOOK 4 SEARCH")
print("=" * 80)

if nb4_candidates:

    for path in nb4_candidates:
        print(path)

    # Prefer production version
    production_candidates = [
        path
        for path in nb4_candidates
        if "production" in os.path.basename(path).lower()
    ]

    if production_candidates:
        NB4_PATH = production_candidates[0]
    else:
        NB4_PATH = nb4_candidates[0]

    print()
    print("Notebook 4 selected:")
    print(NB4_PATH)

else:

    raise FileNotFoundError(
        "Notebook 4 could not be found inside the GitHub repository."
    )

NOTEBOOK 4 SEARCH
/content/ANLOK-WATER-PREDICTION-PROJECT-/NB4_OP.ipynb
/content/ANLOK-WATER-PREDICTION-PROJECT-/NB4_Production.ipynb

Notebook 4 selected:
/content/ANLOK-WATER-PREDICTION-PROJECT-/NB4_Production.ipynb


In [6]:
# ============================================================
# CELL 6 — NB4 OUTPUT PATHS
# ============================================================

MODEL_PATH = os.path.join(
    MODELS_DIR,
    "best_model.joblib"
)

MODELLING_TABLE_PATH = os.path.join(
    MODELLING_DATA_DIR,
    "notebook4_reliability_modelling_table.csv"
)

FEATURE_DICTIONARY_PATH = os.path.join(
    MODEL_EVALUATION_DIR,
    "feature_dictionary.csv"
)

FEATURE_IMPORTANCE_PATH = os.path.join(
    MODEL_EVALUATION_DIR,
    "feature_importance.csv"
)

PERMUTATION_IMPORTANCE_PATH = os.path.join(
    MODEL_EVALUATION_DIR,
    "permutation_importance.csv"
)

FINAL_METRICS_PATH = os.path.join(
    MODEL_EVALUATION_DIR,
    "final_test_metrics.csv"
)

FINAL_CONFIG_PATH = os.path.join(
    MODEL_EVALUATION_DIR,
    "final_model_configuration.json"
)


required_files = {
    "Production model": MODEL_PATH,
    "NB4 modelling table": MODELLING_TABLE_PATH
}


print("=" * 80)
print("NB4 OUTPUT VALIDATION")
print("=" * 80)


for name, path in required_files.items():

    if not os.path.exists(path):

        raise FileNotFoundError(
            f"{name} was not found:\n{path}\n\n"
            "Run Notebook 4 successfully first."
        )

    print(f"✓ {name}")
    print(f"  {path}")


print()
print("Required NB4 files found successfully.")

NB4 OUTPUT VALIDATION
✓ Production model
  /content/ANLOK-WATER-PREDICTION-PROJECT-/Outputs/Models/best_model.joblib
✓ NB4 modelling table
  /content/ANLOK-WATER-PREDICTION-PROJECT-/Outputs/ModellingData/notebook4_reliability_modelling_table.csv

Required NB4 files found successfully.


In [7]:
# ============================================================
# CELL 7 — LOAD NB4 MODEL AND DATA
# ============================================================

model = joblib.load(
    MODEL_PATH
)

modelling_df = pd.read_csv(
    MODELLING_TABLE_PATH
)


print("=" * 80)
print("NB4 MODEL LOADED")
print("=" * 80)

print("Model type:")
print(type(model))

print()
print("Modelling table shape:")
print(modelling_df.shape)

print()
print("Modelling table columns:")

for column in modelling_df.columns:
    print(" -", column)


display(
    modelling_df.head()
)

NB4 MODEL LOADED
Model type:
<class 'sklearn.pipeline.Pipeline'>

Modelling table shape:
(144, 13)

Modelling table columns:
 - wsa_name
 - population
 - population_with_access_to_water
 - urban_population
 - urban_population_with_access_to_water
 - rural_population
 - rural_population_with_access_to_water
 - log_population
 - water_access_rate_pct
 - water_access_gap_pct
 - urban_population_share_pct
 - rural_population_share_pct
 - target_reliable_supply_pct


,wsa_name,population,population_with_access_to_water,urban_population,urban_population_with_access_to_water,rural_population,rural_population_with_access_to_water,log_population,water_access_rate_pct,water_access_gap_pct,urban_population_share_pct,rural_population_share_pct,target_reliable_supply_pct
0,Alfred Nzo,829109.0,323859.0,46614.0,17722.0,782495.0,306137.0,13.628108,39.061088,60.938912,5.622180,94.377820,16.0
1,Amathole,838800.0,509183.0,148028.0,106420.0,690772.0,402763.0,13.639729,60.703743,39.296257,17.647592,82.352408,23.0
2,Blue Crane Route,34291.0,32784.0,26791.0,25604.0,7500.0,7180.0,10.442667,95.605261,4.394739,78.128372,21.871628,86.0
3,Buffalo City Metropolitan Municipality,796675.0,771345.0,465125.0,450270.0,331550.0,321075.0,13.588203,96.820535,3.179465,58.383281,41.616719,72.0
4,Chris Hani,800718.0,596366.0,171148.0,157242.0,629570.0,439124.0,13.593265,74.478905,25.521095,21.374317,78.625683,40.0


In [8]:
# ============================================================
# CELL 8 — VALIDATE NB4 MODEL FEATURES
# ============================================================

EXPECTED_FEATURES = [

    "wsa_name",

    "population",

    "population_with_access_to_water",

    "urban_population",

    "urban_population_with_access_to_water",

    "rural_population",

    "rural_population_with_access_to_water",

    "log_population",

    "water_access_rate_pct",

    "water_access_gap_pct",

    "urban_population_share_pct",

    "rural_population_share_pct"
]


# ------------------------------------------------------------
# Get features expected by fitted model
# ------------------------------------------------------------

if hasattr(model, "feature_names_in_"):

    MODEL_FEATURES = list(
        model.feature_names_in_
    )

else:

    MODEL_FEATURES = EXPECTED_FEATURES.copy()


print("=" * 80)
print("FEATURE VALIDATION")
print("=" * 80)

print()
print("Features expected by NB4 model:")

for feature in MODEL_FEATURES:
    print(" -", feature)


# ------------------------------------------------------------
# Compare with modelling table
# ------------------------------------------------------------

missing_features = [
    feature
    for feature in MODEL_FEATURES
    if feature not in modelling_df.columns
]


unexpected_features = [
    feature
    for feature in EXPECTED_FEATURES
    if feature not in MODEL_FEATURES
]


if missing_features:

    raise ValueError(
        "Missing NB4 features:\n"
        + str(missing_features)
    )


print()
print("Expected NB4 features:", len(MODEL_FEATURES))
print(
    "Available interface features:",
    len(MODEL_FEATURES)
)

print("Feature alignment: PASSED ✓")

FEATURE VALIDATION

Features expected by NB4 model:
 - wsa_name
 - population
 - population_with_access_to_water
 - urban_population
 - urban_population_with_access_to_water
 - rural_population
 - rural_population_with_access_to_water
 - log_population
 - water_access_rate_pct
 - water_access_gap_pct
 - urban_population_share_pct
 - rural_population_share_pct

Expected NB4 features: 12
Available interface features: 12
Feature alignment: PASSED ✓


In [9]:
# ============================================================
# CELL 9 — TEST NB4 PREDICTION
# ============================================================

available_regions = sorted(
    modelling_df["wsa_name"]
    .dropna()
    .astype(str)
    .unique()
)


print(
    "Number of water service areas:",
    len(available_regions)
)

print()
print("First 10:")

for region in available_regions[:10]:
    print(" -", region)


# ------------------------------------------------------------
# Test first available municipality
# ------------------------------------------------------------

test_region = available_regions[0]

test_row = modelling_df[
    modelling_df["wsa_name"].astype(str)
    == test_region
].iloc[0]


features = pd.DataFrame([
    {
        feature: test_row[feature]
        for feature in MODEL_FEATURES
    }
])


prediction = model.predict(
    features
)[0]


predicted_reliability_pct = float(
    np.clip(
        prediction,
        0,
        100
    )
)


outage_risk_pct = (
    100
    - predicted_reliability_pct
)


print()
print("=" * 80)

print("REGION:", test_region)

print(
    "Predicted reliability:",
    round(
        predicted_reliability_pct,
        2
    ),
    "%"
)

print(
    "Outage risk proxy:",
    round(
        outage_risk_pct,
        2
    ),
    "%"
)

Number of water service areas: 144

First 10:
 - !Kheis
 - Alfred Nzo
 - Amajuba
 - Amathole
 - Beaufort West
 - Bela-Bela
 - Bergrivier
 - Bitou
 - Blue Crane Route
 - Breede Valley

REGION: !Kheis
Predicted reliability: 73.85 %
Outage risk proxy: 26.15 %


In [10]:
# ============================================================
# CELL 10 — CREATE FASTAPI BACKEND
# ============================================================

%%writefile main.py

import os
import json

import numpy as np
import pandas as pd
import joblib

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel


# ============================================================
# PROJECT CONFIGURATION
# ============================================================

PROJECT_ROOT = "/content/ANLOK-WATER-PREDICTION-PROJECT-"

MODEL_PATH = os.path.join(
    PROJECT_ROOT,
    "Outputs",
    "Models",
    "best_model.joblib"
)

MODELLING_TABLE_PATH = os.path.join(
    PROJECT_ROOT,
    "Outputs",
    "ModellingData",
    "notebook4_reliability_modelling_table.csv"
)

FEATURE_IMPORTANCE_PATH = os.path.join(
    PROJECT_ROOT,
    "Outputs",
    "ModelEvaluation",
    "permutation_importance.csv"
)

FINAL_METRICS_PATH = os.path.join(
    PROJECT_ROOT,
    "Outputs",
    "ModelEvaluation",
    "final_test_metrics.csv"
)

FINAL_CONFIG_PATH = os.path.join(
    PROJECT_ROOT,
    "Outputs",
    "ModelEvaluation",
    "final_model_configuration.json"
)


# ============================================================
# LOAD NB4 MODEL
# ============================================================

if not os.path.exists(MODEL_PATH):

    raise RuntimeError(
        "NB4 model does not exist. "
        "Run Notebook 4 first."
    )


model = joblib.load(
    MODEL_PATH
)


# ============================================================
# LOAD NB4 MODELLING TABLE
# ============================================================

if not os.path.exists(
    MODELLING_TABLE_PATH
):

    raise RuntimeError(
        "NB4 modelling table could not be found."
    )


data = pd.read_csv(
    MODELLING_TABLE_PATH
)

# ============================================================
# FILTER MODELLING DATA — GAUTENG ONLY
# ============================================================

GAUTENG_WSA_NAMES = {
    "City of Johannesburg Metropolitan Municipality",
    "City of Tshwane Metropolitan Municipality",
    "Ekurhuleni Metropolitan Municipality",
    "Emfuleni",
    "Lesedi",
    "Merafong City",
    "Midvaal",
    "Mogale City",
    "Rand West City",
}

# Keep only Gauteng WSA records present in the dataset.
data["wsa_name"] = data["wsa_name"].astype(str).str.strip()

data = data[
    data["wsa_name"].isin(GAUTENG_WSA_NAMES)
].copy()

if data.empty:
    raise RuntimeError(
        "No Gauteng WSA records were found. "
        "Check the names in the modelling table."
    )

print("ANLOK — Gauteng regions loaded:")
print(sorted(data["wsa_name"].unique()))
print("Total Gauteng regions:", data["wsa_name"].nunique())


# ============================================================
# MODEL FEATURES
# ============================================================

if hasattr(
    model,
    "feature_names_in_"
):

    MODEL_FEATURES = list(
        model.feature_names_in_
    )

else:

    MODEL_FEATURES = [
        column
        for column in data.columns
        if column
        != "target_reliable_supply_pct"
    ]


# ============================================================
# VALIDATE MODEL FEATURES
# ============================================================

missing_features = [
    feature
    for feature in MODEL_FEATURES
    if feature not in data.columns
]


if missing_features:

    raise RuntimeError(
        "Model feature mismatch. "
        f"Missing: {missing_features}"
    )


# ============================================================
# CREATE API
# ============================================================

app = FastAPI(

    title="ANLOK Water Reliability API",

    description=(
        "API powered by the production "
        "Notebook 4 machine-learning model."
    ),

    version="2.0"
)


# ============================================================
# REQUEST MODEL
# ============================================================

class RegionRequest(BaseModel):

    region: str


# ============================================================
# UTILITY
# ============================================================

def python_value(value):

    if pd.isna(value):
        return None

    if isinstance(
        value,
        (
            np.integer,
        )
    ):
        return int(value)

    if isinstance(
        value,
        (
            np.floating,
        )
    ):
        return float(value)

    return value


# ============================================================
# HOME
# ============================================================

@app.get("/")
def home():

    return {

        "project":
        "ANLOK Water Prediction Project",

        "status":
        "running",

        "model_source":
        "Notebook 4 Production Model",

        "target":
        "Water supply reliability percentage"
    }


# ============================================================
# HEALTH
# ============================================================

@app.get("/health")
def health():

    return {

        "status": "healthy",

        "model_loaded": True,

        "records":
        len(data),

        "features":
        len(MODEL_FEATURES)
    }


# ============================================================
# REGIONS
# ============================================================

@app.get("/regions")
def regions():

    values = sorted(
        data["wsa_name"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    return {
        "regions": values
    }


# ============================================================
# REGION DETAILS
# ============================================================

@app.get("/region/{region_name}")
def region_details(
    region_name: str
):

    match = data[
        data["wsa_name"]
        .astype(str)
        .str.strip()
        .str.lower()
        ==
        region_name
        .strip()
        .lower()
    ]


    if match.empty:

        raise HTTPException(
            status_code=404,
            detail="Region not found."
        )


    row = match.iloc[0]


    result = {}

    for column in data.columns:

        result[column] = python_value(
            row[column]
        )


    return result


# ============================================================
# PREDICT
# ============================================================

@app.post("/predict")
def predict(
    request: RegionRequest
):

    match = data[
        data["wsa_name"]
        .astype(str)
        .str.strip()
        .str.lower()
        ==
        request.region
        .strip()
        .lower()
    ]


    if match.empty:

        raise HTTPException(
            status_code=404,
            detail=(
                f"{request.region} "
                "was not found."
            )
        )


    row = match.iloc[0]


    X = pd.DataFrame(
        [
            {
                feature:
                row[feature]

                for feature
                in MODEL_FEATURES
            }
        ]
    )


    # --------------------------------------------------------
    # NB4 REGRESSION PREDICTION
    # --------------------------------------------------------

    raw_prediction = float(
        model.predict(X)[0]
    )


    predicted_reliability = float(
        np.clip(
            raw_prediction,
            0,
            100
        )
    )


    outage_risk = float(
        100
        - predicted_reliability
    )


    # --------------------------------------------------------
    # DASHBOARD INTERPRETATION BANDS
    # --------------------------------------------------------

    if outage_risk >= 70:

        risk_level = "HIGH"

    elif outage_risk >= 40:

        risk_level = "MEDIUM"

    else:

        risk_level = "LOW"


    response = {

        "region":
        str(row["wsa_name"]),

        "predicted_reliability_pct":
        predicted_reliability,

        "outage_risk_pct":
        outage_risk,

        "risk_level":
        risk_level
    }


    # --------------------------------------------------------
    # ADD NB4 FEATURES
    # --------------------------------------------------------

    for feature in MODEL_FEATURES:

        response[feature] = (
            python_value(
                row[feature]
            )
        )


    if (
        "target_reliable_supply_pct"
        in row.index
    ):

        response[
            "observed_reliability_pct"
        ] = python_value(
            row[
                "target_reliable_supply_pct"
            ]
        )


    return response


# ============================================================
# FEATURE IMPORTANCE
# ============================================================

@app.get("/feature-importance")
def feature_importance():

    if not os.path.exists(
        FEATURE_IMPORTANCE_PATH
    ):

        return {
            "features": []
        }


    frame = pd.read_csv(
        FEATURE_IMPORTANCE_PATH
    )


    frame = frame.sort_values(
        "importance_mean",
        ascending=False
    )


    return {

        "features":
        frame.to_dict(
            orient="records"
        )
    }


# ============================================================
# CORRELATION MATRIX
# ============================================================

@app.get("/correlations")
def correlations():

    numeric = data.select_dtypes(
        include=[
            "number"
        ]
    )


    correlation = numeric.corr()


    return {

        "columns":
        correlation.columns.tolist(),

        "matrix":
        correlation.values.tolist()
    }


# ============================================================
# MUNICIPALITY PROFILES
# ============================================================

@app.get("/profiles")
def profiles():

    columns = [

        "wsa_name",

        "water_access_rate_pct",

        "water_access_gap_pct",

        "urban_population_share_pct",

        "rural_population_share_pct"
    ]


    available = [
        column
        for column in columns
        if column in data.columns
    ]


    profile = data[
        available
    ].copy()


    profile = profile.replace(
        {
            np.nan: None
        }
    )


    return {

        "profiles":
        profile.to_dict(
            orient="records"
        )
    }


# ============================================================
# MODEL INFORMATION
# ============================================================

@app.get("/model-info")
def model_info():

    information = {

        "model_features":
        MODEL_FEATURES,

        "number_of_features":
        len(MODEL_FEATURES),

        "observations":
        len(data)
    }


    if os.path.exists(
        FINAL_CONFIG_PATH
    ):

        with open(
            FINAL_CONFIG_PATH,
            "r"
        ) as file:

            configuration = json.load(
                file
            )

        information.update(
            configuration
        )


    if os.path.exists(
        FINAL_METRICS_PATH
    ):

        metrics = pd.read_csv(
            FINAL_METRICS_PATH
        )

        if not metrics.empty:

            information[
                "metrics"
            ] = metrics.iloc[0].to_dict()


    return information

Writing main.py


In [11]:
# ============================================================
# CELL 12 — TEST FASTAPI
# ============================================================

# The FastAPI application needs to be running before testing.
# Starting the FastAPI server in the background using uvicorn and pyngrok.

# Ensure pyngrok is authorized if you haven't done so in this session
# from pyngrok import ngrok
# ngrok.set_auth_token("YOUR_NGROK_AUTH_TOKEN") # Replace with your actual auth token

# Start uvicorn in the background
import subprocess
import threading
import time

def run_server():
    proc = subprocess.Popen(["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"])
    # Store the process object to terminate it later if needed
    global uvicorn_process
    uvicorn_process = proc

# Run the server in a separate thread
server_thread = threading.Thread(target=run_server)
server_thread.start()

# Give the server a moment to start up
time.sleep(5)

import requests


API_URL = (
    "http://127.0.0.1:8000"
)


# ------------------------------------------------------------
# Health
# ------------------------------------------------------------

health = requests.get(
    f"{API_URL}/health"
)

print(
    "Health:",
    health.json()
)


# ------------------------------------------------------------
# Regions
# ------------------------------------------------------------

regions_response = requests.get(
    f"{API_URL}/regions"
)

regions = regions_response.json()[
    "regions"
]


print()
print(
    "Number of regions:",
    len(regions)
)

print(
    "Example:",
    regions[0]
)


# ------------------------------------------------------------
# Prediction
# ------------------------------------------------------------

test_prediction = requests.post(

    f"{API_URL}/predict",

    json={
        "region":
        regions[0]
    }
)


print()
print("Prediction test:")

display(
    test_prediction.json()
)

# Optional: Terminate the uvicorn process after testing if it's no longer needed.
# if 'uvicorn_process' in globals() and uvicorn_process.poll() is None:
#    uvicorn_process.terminate()
#    uvicorn_process.wait()
#    print("Uvicorn server terminated.")

Health: {'status': 'healthy', 'model_loaded': True, 'records': 9, 'features': 12}

Number of regions: 9
Example: City of Johannesburg Metropolitan Municipality

Prediction test:


{'region': 'City of Johannesburg Metropolitan Municipality',
 'predicted_reliability_pct': 78.23977406353833,
 'outage_risk_pct': 21.760225936461666,
 'risk_level': 'LOW',
 'wsa_name': 'City of Johannesburg Metropolitan Municipality',
 'population': 5858600.0,
 'population_with_access_to_water': 5917763.0,
 'urban_population': 5605145.0,
 'urban_population_with_access_to_water': 5661750.0,
 'rural_population': 253455.0,
 'rural_population_with_access_to_water': 256013.0,
 'log_population': 15.583421395849568,
 'water_access_rate_pct': 101.00984876933056,
 'water_access_gap_pct': -1.0098487693305458,
 'urban_population_share_pct': 95.67379578738948,
 'rural_population_share_pct': 4.326204212610521,
 'observed_reliability_pct': 81.0}

In [34]:

import os
import json
import requests

PROJECT_ROOT = "/content/ANLOK-WATER-PREDICTION-PROJECT-"

BOUNDARY_PATH = os.path.join(
    PROJECT_ROOT,
    "Outputs",
    "Interface",
    "gauteng_municipal_boundaries.geojson"
)

os.makedirs(os.path.dirname(BOUNDARY_PATH), exist_ok=True)

# Use the Gauteng Local Municipalities layer (46).
# Do not supply a rectangular geometry filter.
url = (
    "https://ndagis.nda.agric.za/arcgis/rest/services/"
    "Gauteng/MapServer/46/query"
)

params = {
    "where": "1=1",
    "outFields": "*",
    "returnGeometry": "true",
    "outSR": "4326",
    "f": "geojson"
}

response = requests.get(url, params=params, timeout=120)

print("HTTP status:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))

response.raise_for_status()

# Check the response before attempting JSON parsing.
try:
    boundaries = response.json()
except ValueError:
    print("\nResponse preview:")
    print(response.text[:1000])
    raise RuntimeError(
        "The GIS server did not return valid JSON. "
        "Check the response preview above."
    )

if boundaries.get("error"):
    raise RuntimeError(
        f"GIS service error: {boundaries['error']}"
    )

features = boundaries.get("features", [])

if not features:
    raise ValueError(
        "The GIS service returned no boundary polygons."
    )

# Confirm the municipality-name field.
print("\nAvailable fields:")
print(list(features[0].get("properties", {}).keys()))

# The service uses ADM3_EN for municipality names.
# Copy that value into MUNICNAME for compatibility with app.py.
for feature in features:
    props = feature.setdefault("properties", {})
    name = props.get("ADM3_EN")

    if not name:
        raise ValueError(
            "ADM3_EN is missing from a feature. "
            f"Available fields: {list(props.keys())}"
        )

    props["MUNICNAME"] = name

# Save only after successful validation.
boundary_data = {
    "type": "FeatureCollection",
    "features": features
}

with open(BOUNDARY_PATH, "w", encoding="utf-8") as file:
    json.dump(boundary_data, file)

print("\nSaved boundary file:", BOUNDARY_PATH)
print("Number of polygons:", len(features))

# Check whether these are detailed polygons.
print("\nMunicipality boundary detail:")

for feature in features:
    props = feature.get("properties", {})
    geometry = feature.get("geometry", {})
    coords = geometry.get("coordinates", [])

    if geometry.get("type") == "Polygon":
        point_count = sum(len(ring) for ring in coords)
    elif geometry.get("type") == "MultiPolygon":
        point_count = sum(
            len(ring)
            for polygon in coords
            for ring in polygon
        )
    else:
        point_count = 0

    print(
        props["MUNICNAME"],
        "|",
        geometry.get("type"),
        "| points:",
        point_count
    )


HTTP status: 200
Content type: application/geo+json; charset=UTF-8

Available fields:
['Shape_Length', 'Shape_Area', 'OBJECTID', 'ADM3_EN', 'ADM2_EN', 'ADM1_EN']

Saved boundary file: /content/ANLOK-WATER-PREDICTION-PROJECT-/Outputs/Interface/gauteng_municipal_boundaries.geojson
Number of polygons: 9

Municipality boundary detail:
City of Johannesburg | Polygon | points: 2845
City of Tshwane | Polygon | points: 2153
Ekurhuleni | Polygon | points: 2059
Emfuleni | Polygon | points: 1713
Lesedi | Polygon | points: 1854
Merafong City | Polygon | points: 1372
Midvaal | Polygon | points: 2617
Mogale City | Polygon | points: 1907
Rand West City | Polygon | points: 1076


In [13]:

%pip install streamlit-folium folium


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.9/536.9 kB 9.2 MB/s eta 0:00:00


In [35]:
# ============================================================
# CELL 13 — COMPLETE ANLOK SINGLE-PAGE DASHBOARD
# ============================================================

%%writefile app.py

# ============================================================
# IMPORTS
# ============================================================

import os
import html as html_lib

import pandas as pd
import requests
import plotly.express as px

import streamlit as st
import streamlit.components.v1 as components

import plotly.graph_objects as go

# ============================================================
# PAGE CONFIGURATION
# ============================================================

st.set_page_config(
    page_title="ANLOK Water",
    page_icon="💧",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ANLOK WATER — animated background layer
st.markdown("""
<style>
/* Let the animated layer show through */
.stApp,
[data-testid="stAppViewContainer"],
[data-testid="stMain"],
[data-testid="stMainBlockContainer"],
[data-testid="stHeader"] {
    background-color: transparent !important;
}

/* Animated lighter blue background */
#anlok-animated-background {
    position: fixed;
    inset: 0;
    width: 100vw;
    height: 100vh;
    pointer-events: none;
    z-index: -1;

    background:
        radial-gradient(
            circle at 15% 20%,
            rgba(0, 225, 255, 0.32),
            transparent 38%
        ),
        radial-gradient(
            circle at 85% 75%,
            rgba(0, 130, 255, 0.30),
            transparent 42%
        ),
        linear-gradient(
            120deg,
            #061a2b,
            #0b3b5a,
            #08253f,
            #124968
        );

    background-size: 200% 200%, 200% 200%, 300% 300%;
    animation: anlokBackgroundFlow 14s ease-in-out infinite alternate;
}

/* Moving glowing cyber grid */
#anlok-animated-background::after {
    content: "";
    position: absolute;
    inset: 0;

    background-image:
        linear-gradient(
            rgba(0, 220, 255, 0.09) 1px,
            transparent 1px
        ),
        linear-gradient(
            90deg,
            rgba(0, 220, 255, 0.09) 1px,
            transparent 1px
        );

    background-size: 45px 45px;
    animation: anlokGridFlow 20s linear infinite;
}

/* Flowing gradient animation */
@keyframes anlokBackgroundFlow {
    0% {
        background-position: 0% 0%, 100% 100%, 0% 50%;
    }
    100% {
        background-position: 100% 100%, 0% 0%, 100% 50%;
    }
}

/* Slowly moving grid */
@keyframes anlokGridFlow {
    from {
        transform: translate(0, 0);
    }
    to {
        transform: translate(45px, 45px);
    }
}

/* Keep dashboard content above the background */
[data-testid="stAppViewContainer"] {
    position: relative;
}

/* ========================================
   ANLOK ANIMATED SIDEBAR
   Keep inside your existing <style> block
======================================== */

/* Sidebar base */
[data-testid="stSidebar"] {
    background: linear-gradient(
        160deg,
        rgba(5, 27, 45, 0.97),
        rgba(8, 49, 75, 0.96),
        rgba(4, 29, 49, 0.98)
    ) !important;

    border-right: 1px solid rgba(0, 220, 255, 0.25);
    box-shadow: 4px 0 25px rgba(0, 160, 255, 0.10);

    background-size: 200% 200%;
    animation: anlokSidebarFlow 12s ease-in-out infinite alternate;
}

/* Keep sidebar content transparent to show the gradient */
[data-testid="stSidebar"] > div:first-child {
    background: transparent !important;
}

/* Slowly shifting sidebar colours */
@keyframes anlokSidebarFlow {
    0% {
        background-position: 0% 0%;
        box-shadow: 4px 0 20px rgba(0, 160, 255, 0.10);
    }

    100% {
        background-position: 100% 100%;
        box-shadow: 4px 0 30px rgba(0, 210, 255, 0.18);
    }
}

/* Sidebar navigation and interactive elements */
[data-testid="stSidebar"] [data-testid="stMarkdownContainer"],
[data-testid="stSidebar"] label {
    transition: color 0.25s ease, text-shadow 0.25s ease;
}

/* Subtle cyan glow when hovering over buttons */
[data-testid="stSidebar"] .stButton button {
    border: 1px solid rgba(0, 220, 255, 0.30);
    background: rgba(0, 130, 200, 0.12);
    transition: all 0.25s ease;
}

[data-testid="stSidebar"] .stButton button:hover {
    border-color: rgba(0, 230, 255, 0.85);
    background: rgba(0, 180, 240, 0.20);
    box-shadow: 0 0 14px rgba(0, 210, 255, 0.22);
    transform: translateY(-1px);
}

/* Highlight sidebar select boxes on focus */
[data-testid="stSidebar"] [data-baseweb="select"]:focus-within {
    box-shadow: 0 0 12px rgba(0, 210, 255, 0.25);
    border-radius: 8px;
}

</style>

<div id="anlok-animated-background"></div>
""", unsafe_allow_html=True)



# ============================================================
# CONFIGURATION
# ============================================================

API_URL = "http://127.0.0.1:8000"

PROJECT_ROOT = (
    "/content/"
    "ANLOK-WATER-PREDICTION-PROJECT-"
)

COORDINATES_PATH = os.path.join(
    PROJECT_ROOT,
    "Outputs",
    "Interface",
    "wsa_coordinates.csv"
)


# ============================================================
# GLOBAL CSS
# ============================================================

# Animated background CSS — background layers stay behind all dashboard content
st.markdown("""
<style>
html {
    scroll-behavior: smooth;
}


@keyframes anlokGlowDrift {
    0%   { background-position: 0% 20%; transform: translate3d(-1%, -1%, 0) scale(1); }
    50%  { background-position: 70% 50%; transform: translate3d(1%, 1%, 0) scale(1.03); }
    100% { background-position: 100% 80%; transform: translate3d(-1%, 2%, 0) scale(1.06); }
}

@keyframes anlokGridMove {
    from { background-position: 0 0, 0 0; }
    to   { background-position: 0 48px, 48px 0; }
}

/* Keep actual Streamlit content above the background layers. */
[data-testid="stAppViewContainer"] {
    position: relative !important;
    z-index: 1 !important;
    background: transparent !important;
}

[data-testid="stMain"],
[data-testid="stMainBlockContainer"],
[data-testid="stVerticalBlock"],
[data-testid="stHorizontalBlock"] {
    background: transparent !important;
}

[data-testid="stMainBlockContainer"] {
    position: relative !important;
    z-index: 2 !important;
    max-width: 1450px;
    padding-top: 2rem;
    padding-bottom: 5rem;
}


/* ============================================
   ANLOK — ANIMATED SIDEBAR
============================================ */

section[data-testid="stSidebar"] {
    position: relative !important;
    z-index: 10 !important;

    background: linear-gradient(
        145deg,
        #0b3554 0%,
        #10577b 35%,
        #0a4168 65%,
        #12638a 100%
    ) !important;

    background-size: 300% 300% !important;
    animation: anlokSidebarFlow 10s ease-in-out infinite alternate !important;

    border-right: 1px solid rgba(0, 225, 255, 0.55) !important;

    box-shadow:
        4px 0 20px rgba(0, 190, 255, 0.12),
        inset -1px 0 12px rgba(0, 225, 255, 0.06);
}

/* Animate the sidebar gradient and glow */
@keyframes anlokSidebarFlow {
    0% {
        background-position: 0% 0%;
        box-shadow:
            4px 0 18px rgba(0, 190, 255, 0.12),
            inset -1px 0 8px rgba(0, 225, 255, 0.05);
    }

    50% {
        background-position: 50% 100%;
        box-shadow:
            4px 0 28px rgba(0, 210, 255, 0.22),
            inset -1px 0 14px rgba(0, 225, 255, 0.09);
    }

    100% {
        background-position: 100% 0%;
        box-shadow:
            4px 0 35px rgba(0, 225, 255, 0.30),
            inset -1px 0 18px rgba(0, 225, 255, 0.12);
    }
}

/* Sidebar text */
section[data-testid="stSidebar"] * {
    color: #f4fbff;
}

/* ============================================
   SIDEBAR LOGO GLOW
============================================ */

.sidebar-logo {
    font-size: 27px;
    font-weight: 900;
    letter-spacing: 3px;
    text-align: center;
    color: #ffffff !important;

    text-shadow:
        0 0 5px rgba(0, 225, 255, 0.65),
        0 0 14px rgba(0, 190, 255, 0.45);

    animation: anlokLogoGlow 3s ease-in-out infinite alternate;
}

.sidebar-subtitle {
    text-align: center;
    color: #a9eaff !important;
    font-size: 12px;
    letter-spacing: 1px;
    margin-top: 8px;
    margin-bottom: 24px;
}

@keyframes anlokLogoGlow {
    from {
        text-shadow: 0 0 5px rgba(0, 225, 255, 0.35);
    }

    to {
        text-shadow:
            0 0 10px rgba(0, 225, 255, 0.9),
            0 0 22px rgba(0, 190, 255, 0.65);
    }
}

/* ============================================
   ANIMATED SIDEBAR NAVIGATION
============================================ */

.nav-container {
    display: flex;
    flex-direction: column;
    gap: 10px;
    margin: 18px 0 24px 0;
}

.anlok-nav-link {
    position: relative;
    display: flex;
    align-items: center;
    gap: 13px;

    padding: 14px 15px;
    overflow: hidden;

    text-decoration: none !important;
    color: #d7f3ff !important;

    background: rgba(5, 31, 52, 0.35);
    border: 1px solid rgba(130, 220, 255, 0.12);
    border-radius: 11px;

    font-size: 14px;
    font-weight: 650;

    transition:
        background 0.3s ease,
        border-color 0.3s ease,
        box-shadow 0.3s ease,
        transform 0.3s ease;
}

/* Moving light sweep */
.anlok-nav-link::before {
    content: "";
    position: absolute;
    top: 0;
    left: -130%;
    width: 75%;
    height: 100%;

    background: linear-gradient(
        110deg,
        transparent,
        rgba(0, 225, 255, 0.16),
        transparent
    );

    transition: left 0.55s ease;
    pointer-events: none;
}

.anlok-nav-link:hover::before {
    left: 150%;
}

.anlok-nav-link:hover {
    color: #ffffff !important;
    background: rgba(0, 174, 230, 0.17);
    border-color: rgba(0, 225, 255, 0.65);

    box-shadow:
        0 0 12px rgba(0, 210, 255, 0.16),
        inset 0 0 12px rgba(0, 210, 255, 0.06);

    transform: translateX(3px);
}

/* Active navigation item */
.anlok-nav-link.active {
    color: #ffffff !important;

    background: linear-gradient(
        100deg,
        rgba(0, 174, 230, 0.30),
        rgba(0, 90, 150, 0.20)
    );

    border-color: rgba(0, 230, 255, 0.85);

    box-shadow:
        0 0 15px rgba(0, 210, 255, 0.20),
        inset 3px 0 0 #27e5ff;
}

/* Glowing navigation indicator */
.anlok-nav-link.active::after {
    content: "";
    position: absolute;
    right: 10px;
    top: 50%;

    width: 6px;
    height: 6px;
    border-radius: 50%;

    background: #51f3ff;
    box-shadow:
        0 0 6px #51f3ff,
        0 0 14px #00cfff;

    transform: translateY(-50%);
    animation: anlokIndicatorPulse 1.8s ease-in-out infinite;
}

.nav-icon {
    color: #67e8f9 !important;
    font-size: 20px;
    min-width: 22px;
    text-align: center;
    text-shadow: 0 0 8px rgba(0, 225, 255, 0.5);
}

/* Sidebar information panel */
.sidebar-info {
    margin-top: 18px;
    padding: 16px;

    color: #e3f6ff !important;
    background: rgba(3, 25, 44, 0.55);

    border: 1px solid rgba(0, 210, 255, 0.30);
    border-radius: 12px;

    box-shadow: inset 0 0 16px rgba(0, 160, 255, 0.05);
}

.sidebar-info strong {
    color: #7eeeff !important;
}

/* Accessibility: respect reduced-motion preferences */
@media (prefers-reduced-motion: reduce) {
    section[data-testid="stSidebar"],
    .sidebar-logo,
    .anlok-nav-link.active::after {
        animation: none !important;
    }

    .anlok-nav-link,
    .anlok-nav-link::before {
        transition: none !important;
    }
}

h1, h2, h3, h4, p, label, li,
[data-testid="stMarkdownContainer"] {
    color: #f4fbff;
}
h1 {
    text-align: center;
    font-weight: 900;
    text-shadow: 0 0 12px rgba(255, 255, 255, .22);
}
h2 { font-weight: 800; }
h3 { font-weight: 700; }

[data-testid="stAlert"],
[data-testid="stSelectbox"],
[data-testid="stButton"],
[data-testid="stPlotlyChart"],
[data-testid="stMetric"] {
    visibility: visible !important;
    opacity: 1 !important;
}

/* Respect accessibility settings for users who prefer less motion. */
@media (prefers-reduced-motion: reduce) {
    .stApp::before,
    .stApp::after {
        animation: none !important;
    }
}


.regional-list {
    display: flex;
    flex-direction: column;
    gap: 10px;
    margin-top: 16px;
}

.regional-list-item {
    display: flex;
    align-items: center;
    justify-content: space-between;
    gap: 14px;
    padding: 14px 16px;
    background: linear-gradient(
        110deg,
        rgba(5, 31, 54, 0.96),
        rgba(9, 60, 88, 0.90)
    );
    border: 1px solid rgba(103, 232, 249, 0.24);
    border-left: 4px solid #38bdf8;
    border-radius: 10px;
    transition: all 0.25s ease;
}

.regional-list-item:hover {
    background: rgba(10, 72, 104, 0.96);
    border-color: rgba(103, 232, 249, 0.7);
    box-shadow: 0 0 16px rgba(56, 189, 248, 0.18);
    transform: translateX(3px);
}

.regional-list-label {
    display: flex;
    align-items: center;
    gap: 10px;
    color: #c8e8f7;
    font-size: 14px;
    font-weight: 600;
}

.regional-list-icon {
    display: flex;
    align-items: center;
    justify-content: center;
    min-width: 34px;
    height: 34px;
    background: rgba(56, 189, 248, 0.13);
    border: 1px solid rgba(103, 232, 249, 0.2);
    border-radius: 9px;
    font-size: 18px;
}

.regional-list-value {
    color: #f0fbff;
    font-size: 15px;
    font-weight: 800;
    text-align: right;
    overflow-wrap: anywhere;
}

/* ==========================================
   REGIONAL SNAPSHOT & SUMMARY CARD CONTAINER
========================================== */

.regional-card {
    margin: 18px 0 28px 0;
    padding: 22px;
    border: 1px solid rgba(103, 232, 249, 0.35);
    border-radius: 16px;
    background: linear-gradient(
        145deg,
        rgba(3, 22, 42, 0.94),
        rgba(8, 59, 88, 0.91)
    );
    box-shadow: 0 8px 28px rgba(0, 0, 0, 0.22);
}

.regional-card-header {
    display: flex;
    align-items: center;
    gap: 14px;
    padding-bottom: 18px;
    margin-bottom: 6px;
    border-bottom: 1px solid rgba(103, 232, 249, 0.25);
}

.regional-card-main-icon {
    display: flex;
    align-items: center;
    justify-content: center;
    width: 48px;
    height: 48px;
    flex-shrink: 0;
    border: 1px solid rgba(103, 232, 249, 0.35);
    border-radius: 12px;
    background: rgba(56, 189, 248, 0.14);
    font-size: 24px;
}

.regional-card-heading {
    min-width: 0;
}

.regional-card-title {
    color: #f0fbff;
    font-size: 21px;
    font-weight: 800;
}

.regional-card-region {
    color: #67e8f9;
    font-size: 15px;
    font-weight: 700;
    margin-top: 4px;
}

.regional-card-subtitle {
    color: #a8c9dc;
    font-size: 13px;
    margin-top: 5px;
}

/* Make the existing list rows fill the card */
.regional-card .regional-list {
    margin-top: 16px;
}

.regional-card .regional-list-item {
    width: 100%;
    box-sizing: border-box;
}

@media (max-width: 600px) {
    .regional-card {
        padding: 14px;
    }

    .regional-card-title {
        font-size: 18px;
    }
}


/* ==========================================
   ANLOK — DARK DONUT CHART CARDS
========================================== */

.st-key-pie_population_access,
.st-key-pie_urban_rural,
.st-key-pie_access_coverage,
.st-key-pie_reliability_risk {
    background: #071827 !important;
    border: 1px solid #243B4A !important;
    border-radius: 16px !important;
    padding: 12px !important;
    box-shadow: none !important;
}

.st-key-pie_population_access [data-testid="stVerticalBlockBorderWrapper"],
.st-key-pie_urban_rural [data-testid="stVerticalBlockBorderWrapper"],
.st-key-pie_access_coverage [data-testid="stVerticalBlockBorderWrapper"],
.st-key-pie_reliability_risk [data-testid="stVerticalBlockBorderWrapper"] {
    background: #071827 !important;
    border: 1px solid #243B4A !important;
    border-radius: 14px !important;
    box-shadow: none !important;
}

.st-key-pie_population_access [data-testid="stPlotlyChart"],
.st-key-pie_urban_rural [data-testid="stPlotlyChart"],
.st-key-pie_access_coverage [data-testid="stPlotlyChart"],
.st-key-pie_reliability_risk [data-testid="stPlotlyChart"] {
    background: transparent !important;
    border: none !important;
    box-shadow: none !important;
}


/* Reliability vs Risk pie chart card */
.st-key-pie_dashboard_risk {
    background: #071827 !important;
    border: 1px solid #243B4A !important;
    border-radius: 16px !important;
    padding: 16px !important;
    box-shadow: none !important;
}

.st-key-pie_dashboard_risk [data-testid="stVerticalBlockBorderWrapper"] {
    background: #071827 !important;
    border: 1px solid #243B4A !important;
    border-radius: 16px !important;
    box-shadow: none !important;
}

.st-key-pie_dashboard_risk [data-testid="stPlotlyChart"] {
    background: #071827 !important;
    border: none !important;
    box-shadow: none !important;
}


/* ==========================================
   ANLOK SIDEBAR — FIX SCROLLING
========================================== */

section[data-testid="stSidebar"] {
    height: 100vh !important;
    max-height: 100vh !important;
    overflow: hidden !important;
}

/* Use Streamlit's actual sidebar content area */
section[data-testid="stSidebar"] > div {
    height: 100vh !important;
    max-height: 100vh !important;
    overflow-y: auto !important;
    overflow-x: hidden !important;
    box-sizing: border-box !important;
    scrollbar-width: thin !important;
    scrollbar-color: #22d3ee #08253f !important;
}

/* Avoid forcing a second, nested scrollbar */
section[data-testid="stSidebar"] [data-testid="stSidebarContent"] {
    min-height: 0 !important;
    height: auto !important;
    max-height: none !important;
    overflow: visible !important;
}

/* Keep all navigation items reachable */
.nav-container {
    padding-bottom: 45px !important;
}

/* Water Access link uses the same layout */
.anlok-nav-link {
    display: flex !important;
    align-items: center !important;
    gap: 13px !important;
    width: 100% !important;
    box-sizing: border-box !important;
    text-decoration: none !important;
}

/* Blue circular navigation indicator */
.anlok-nav-link .nav-icon {
    display: inline-flex !important;
    align-items: center !important;
    justify-content: center !important;
    width: 10px !important;
    height: 10px !important;
    min-width: 10px !important;
    border-radius: 50% !important;
    background: #22d3ee !important;
    box-shadow: 0 0 8px rgba(34, 211, 238, 0.8) !important;
}


</style>
""", unsafe_allow_html=True)


# ============================================================
# API HELPERS
# ============================================================

def api_get(
    endpoint
):

    response = requests.get(
        f"{API_URL}{endpoint}",
        timeout=20
    )

    response.raise_for_status()

    return response.json()


def api_post(
    endpoint,
    payload
):

    response = requests.post(
        f"{API_URL}{endpoint}",
        json=payload,
        timeout=20
    )

    response.raise_for_status()

    return response.json()

# ============================================================
# Regional lists and summary
# ============================================================

def render_regional_list(items):
    html = '<div class="regional-list">'

    for icon, label, value in items:
        html += f"""
        <div class="regional-list-item">
            <div class="regional-list-label">
                <div class="regional-list-icon">{icon}</div>
                <span>{label}</span>
            </div>
            <div class="regional-list-value">{value}</div>
        </div>
        """

    html += "</div>"

    st.markdown(html, unsafe_allow_html=True)

# ============================================================
# CHART INTERPRETATION BAR
# ============================================================

def chart_interpretation(
    text
):

    safe_text = html_lib.escape(
        str(text)
    )

    interpretation_html = f"""
<div class="chart-interpretation">

    <div class="chart-interpretation-title">
         Interpretation
    </div>

    <div>
        {safe_text}
    </div>

</div>
"""

    st.html(
        interpretation_html
    )

# ============================================================
# METRIC CARD
# ============================================================

def metric_card(
    title,
    value,
    subtitle=""
):

    safe_title = html_lib.escape(
        str(title)
    )

    safe_value = html_lib.escape(
        str(value)
    )

    safe_subtitle = html_lib.escape(
        str(subtitle)
    )


    card_html = f"""
<div style="
    background:
        rgba(
            8,
            59,
            92,
            0.26
        );

    border:
        1px solid
        rgba(
            255,
            255,
            255,
            0.24
        );

    border-radius:
        16px;

    padding:
        19px;

    min-height:
        125px;

    box-shadow:
        0 8px 25px
        rgba(
            0,
            40,
            70,
            0.17
        );
">

<div style="
    color:#e3f6ff;
    font-size:13px;
    font-weight:700;
    margin-bottom:8px;
">
{safe_title}
</div>


<div style="
    color:white;
    font-size:30px;
    font-weight:900;
">
{safe_value}
</div>


<div style="
    color:#d4f1ff;
    font-size:12px;
    margin-top:7px;
">
{safe_subtitle}
</div>

</div>
"""

    st.html(
        card_html
    )


# ============================================================
# GENERIC STACKED BLUE CARD
# Uses the regional-list CSS classes
# ============================================================

def render_stacked_blue_card(
    region_name,
    subtitle,
    items
):
    safe_region = html_lib.escape(str(region_name))
    safe_subtitle = html_lib.escape(str(subtitle))

    rows_html = ""

    for item in items:
        number = html_lib.escape(str(item.get("number", "")))
        label = html_lib.escape(str(item.get("label", "")))
        value = html_lib.escape(str(item.get("value", "")))

        # Use the item number as the row icon.
        rows_html += f"""
        <div class="regional-list-item">
            <div class="regional-list-label">
                <div class=>{number}</div>
                <span>{label}</span>
            </div>
            <div class="regional-list-value">{value}</div>
        </div>
        """

    card_html = f"""
    <div class="regional-card">

        <div class="regional-card-header">

            <div class="regional-card-heading">
                <div class="regional-card-region">{safe_region}</div>
                <div class="regional-card-subtitle">
                    {safe_subtitle} {safe_region}
                </div>
            </div>
        </div>

        <div class="regional-list">
            {rows_html}
        </div>

    </div>
    """

    #st.markdown(card_html, unsafe_allow_html=True)
    st.html(card_html)

# ============================================================
# REGIONAL SNAPSHOT CARD
# ============================================================

def render_regional_snapshot_card(
    region_name,
    items
):

    render_stacked_blue_card(
        region_name=region_name,
        subtitle=(
            "Quick water reliability profile for "
        ),
        items=items
    )


# ============================================================
# REGIONAL SUMMARY CARD
# ============================================================

def render_regional_summary_card(
    region_name,
    items
):

    render_stacked_blue_card(
        region_name=region_name,
        subtitle=(
            "Summary profile for "
        ),
        items=items
    )


# ============================================================
# SECTION ANCHOR
# ============================================================

def section_anchor(
    section_id,
    label
):

    safe_section_id = (
        html_lib.escape(
            str(section_id)
        )
    )

    safe_label = (
        html_lib.escape(
            str(label)
        )
    )


    st.html(
        f"""
<div
    id="{safe_section_id}"
    data-scroll-section="{safe_section_id}"
    class="page-section"
>
</div>

<div class="section-tag">
    {safe_label}
</div>
"""
    )


# ============================================================
# CHECK FASTAPI
# ============================================================

try:

    health = api_get(
        "/health"
    )


except Exception as error:

    st.error(
        "FastAPI backend is not running."
    )

    st.write(
        error
    )

    st.stop()


# ============================================================
# LOAD REGIONS
# ============================================================

try:

    regions_response = api_get(
        "/regions"
    )

    regions = regions_response.get(
        "regions",
        []
    )


except Exception as error:

    st.error(
        "Unable to load Water Service Areas."
    )

    st.write(
        error
    )

    st.stop()


if not regions:

    st.error(
        "No Water Service Areas "
        "were returned by the API."
    )

    st.stop()


# ============================================================
# SIDEBAR LOGO
# ============================================================

with st.sidebar:

    st.html(
        """
<div class="sidebar-logo">
 ANLOK WATER
</div>

<div class="sidebar-subtitle">
Water Reliability Intelligence
</div>
"""
    )


# ============================================================
# MAIN TITLE
# ============================================================

st.title(
    " ANLOK WATER"
)


st.markdown(
    """
<div style="
    text-align:center;
    color:#f0fbff;
    font-size:18px;
    margin-bottom:28px;
">

South African Water Reliability
& Risk Intelligence Dashboard

</div>
""",
    unsafe_allow_html=True
)


# ============================================================
# REGION SELECTION
# ============================================================

st.html(
    """
<div class="welcome-card">

<div style="
    font-size:26px;
    font-weight:900;
    color:white;
    margin-bottom:9px;
">

Select a Water Service Area


"""
)


selected_region = st.selectbox(
    "Choose a Water Service Area",
    regions,
    index=None,
    placeholder=(
        "Select a Water Service Area..."
    ),
    key="anlok_region_selector"
)


# ============================================================
# WAIT FOR REGION SELECTION
# ============================================================

if selected_region is None:

    with st.sidebar:

        st.html(
            """
<div class="sidebar-info">

<strong>
Choose a region first
</strong>

<br><br>

Once a Water Service Area is selected,
the complete dashboard and navigation
will become available.

</div>
"""
        )


    st.stop()



# ============================================================
# SIDEBAR NAVIGATION
# ============================================================

with st.sidebar:

    navigation_html = """
    <div class="nav-container">

        <a href="#dashboard-section"
           class="anlok-nav-link"
           data-target="dashboard-section">
            <span class="nav-icon">●</span>
            <span>Dashboard</span>
        </a>

        <a href="#conditions-section"
           class="anlok-nav-link"
           data-target="conditions-section">
            <span class="nav-icon">●</span>
            <span>Regional Conditions</span>
        </a>

         <a href="#water-access-section"
           class="anlok-nav-link"
           data-target="water-access-section">
            <span class="nav-icon">●</span>
            <span>Water Access Locations</span>
        </a>

        <a href="#features-section"
           class="anlok-nav-link"
           data-target="features-section">
            <span class="nav-icon">●</span>
            <span>Feature Analysis</span>
        </a>

        <a href="#map-section"
           class="anlok-nav-link"
           data-target="map-section">
            <span class="nav-icon">●</span>
            <span>Regional Map</span>
        </a>



    </div>
    """

    st.html(navigation_html)

    st.html(
        f"""
        <div class="sidebar-info">
            <strong>Selected Region</strong>
            <br><br>
            {html_lib.escape(str(selected_region))}
            <br><br>
            Use the navigation above to jump
            directly to a dashboard section.
        </div>
        """
    )



# ============================================================
# GET PREDICTION
# ============================================================

try:

    result = api_post(
        "/predict",
        {
            "region":
                selected_region
        }
    )


except Exception as error:

    st.error(
        "Unable to generate a prediction "
        "for the selected region."
    )

    st.write(
        error
    )

    st.stop()


# ============================================================
# VALUES
# ============================================================

reliability = float(
    result.get(
        "predicted_reliability_pct",
        0
    )
)


risk = float(
    result.get(
        "outage_risk_pct",
        0
    )
)


risk_level = str(
    result.get(
        "risk_level",
        "UNKNOWN"
    )
)


population = float(
    result.get(
        "population",
        0
    )
)


population_with_access = float(
    result.get(
        "population_with_access_to_water",
        0
    )
)


population_without_access = max(
    population
    -
    population_with_access,
    0
)


urban_population = float(
    result.get(
        "urban_population",
        0
    )
)


rural_population = float(
    result.get(
        "rural_population",
        0
    )
)


water_access_rate = float(
    result.get(
        "water_access_rate_pct",
        0
    )
)


water_access_gap = float(
    result.get(
        "water_access_gap_pct",
        0
    )
)


urban_share = float(
    result.get(
        "urban_population_share_pct",
        0
    )
)


rural_share = float(
    result.get(
        "rural_population_share_pct",
        0
    )
)


# ============================================================
# SECTION 1 — DASHBOARD
# ============================================================

section_anchor(
    "dashboard-section",
    "DASHBOARD"
)


st.header(
    f" {selected_region}"
)


# ============================================================
# METRIC CARDS
# ============================================================

col1, col2, col3, col4 = (
    st.columns(
        4
    )
)


with col1:

    metric_card(
        "Population",
        f"{population:,.0f}",
        "Total population"
    )


with col2:

    metric_card(
        "Water Access Rate",
        f"{water_access_rate:.1f}%",
        "Population with water access"
    )


with col3:

    metric_card(
        "Predicted Reliability",
        f"{reliability:.1f}%",
        "NB4 model prediction"
    )


with col4:

    metric_card(
        "Outage Risk Proxy",
        f"{risk:.1f}%",
        f"{risk_level} risk"
    )


# ============================================================
# WATER RELIABILITY ASSESSMENT
# ============================================================

st.subheader(
    "Water Reliability Assessment"
)


if (
    risk_level.upper()
    == "HIGH"
):

    st.error(
        f" HIGH RISK — "
        f"{risk:.1f}%"
    )


elif (
    risk_level.upper()
    == "MEDIUM"
):

    st.warning(
        f" MEDIUM RISK — "
        f"{risk:.1f}%"
    )


else:

    st.success(
        f" LOW RISK — "
        f"{risk:.1f}%"
    )


st.caption(
    "The outage-risk value is calculated "
    "as 100 − predicted water-supply "
    "reliability. It is a risk proxy rather "
    "than a directly observed outage probability."
)

# ============================================================
# DASHBOARD — RELIABILITY VS RISK + INTERPRETATION
# ============================================================

dashboard_risk_df = pd.DataFrame({
    "Measure": [
        "Predicted Reliability",
        "Outage Risk Proxy"
    ],
    "Percentage": [
        reliability,
        risk
    ]
})

labels = dashboard_risk_df["Measure"].tolist()
values = dashboard_risk_df["Percentage"].tolist()

# Build the compact donut
dashboard_risk_fig = go.Figure()

# ============================================================
# LAYER 1 — DARK SIDEWALL / DEPTH
# ============================================================

dashboard_risk_fig.add_trace(
    go.Pie(
        labels=labels,
        values=values,
        hole=0.66,
        sort=False,
        direction="clockwise",
        rotation=90,
        marker=dict(
            colors=["#034F4B", "#125B7A"],
            line=dict(color="rgba(0,0,0,0)", width=0)
        ),
        textinfo="none",
        hoverinfo="skip",
        showlegend=False,
        domain=dict(x=[0.05, 0.95], y=[0.08, 0.92])
    )
)

# ============================================================
# LAYER 2 — MAIN COLOURED DONUT
# ============================================================
dashboard_risk_fig.add_trace(
    go.Pie(
        labels=labels,
        values=values,
        hole=0.66,
        sort=False,
        direction="clockwise",
        rotation=90,
        marker=dict(
            colors=["#05716C", "#1978A5"],
            line=dict(color="rgba(0,0,0,0)", width=0)
        ),
        textposition="inside",
        textinfo="percent",
        textfont=dict(
            color="#FFFFFF",
            size=13
        ),
        hovertemplate=(
            "<b>%{label}</b><br>"
            "%{percent}<extra></extra>"
        ),
        showlegend=False,
        domain=dict(x=[0.05, 0.95], y=[0.12, 0.96])
    )
)

dashboard_risk_fig.update_layout(
    height=350,

    title=dict(
        text="Predicted Reliability vs Outage Risk",
        font=dict(
            color="#F1F5F9",
            size=15
        ),
        x=0.5,
        xanchor="center"
    ),

    # Transparent chart background
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",

    font=dict(
        color="#E2E8F0",
        size=11
    ),

    margin=dict(
        l=5,
        r=5,
        t=48,
        b=5
    ),

    showlegend=False
)


# Dynamic interpretation based on the actual model result
if risk_level.upper() == "HIGH":
    dashboard_message = (
        f"{selected_region} has a predicted reliability "
        f"of {reliability:.1f}% and an outage-risk proxy "
        f"of {risk:.1f}%. This indicates high estimated "
        f"vulnerability to unreliable water supply."
    )
elif risk_level.upper() == "MEDIUM":
    dashboard_message = (
        f"{selected_region} has a predicted reliability "
        f"of {reliability:.1f}% and an outage-risk proxy "
        f"of {risk:.1f}%. This indicates moderate estimated "
        f"water-supply risk."
    )
else:
    dashboard_message = (
        f"{selected_region} has a predicted reliability "
        f"of {reliability:.1f}% and an outage-risk proxy "
        f"of {risk:.1f}%. This indicates relatively low "
        f"estimated water-supply risk."
    )

# One native dark panel containing both columns
with st.container(
    key="pie_dashboard_risk",
    border=True
):
    chart_col, interpretation_col = st.columns(
        [1.15, 0.85],
        gap="medium",
        vertical_alignment="center"
    )

    with chart_col:
        st.plotly_chart(
            dashboard_risk_fig,
            use_container_width=True,
            config={"displayModeBar": False},
            key="dashboard_reliability_chart"
        )

    with interpretation_col:
        st.markdown(
            """
            <div style="
                color:#5EEAD4;
                font-size:17px;
                font-weight:800;
                margin-bottom:14px;
            ">
                Risk Interpretation
            </div>
            """,
            unsafe_allow_html=True
        )

        st.markdown(
            f"""
            <div style="
                color:#CBD5E1;
                font-size:14px;
                line-height:1.8;
            ">
                <p>
                    <b style="color:#2DD4BF;">
                        ● Predicted Reliability
                    </b><br>
                    {reliability:.1f}% estimated water-supply
                    reliability for this region.
                </p>
                <p>
                    <b style="color:#FB8067;">
                        ● Outage Risk Proxy
                    </b><br>
                    {risk:.1f}% estimated risk proxy,
                    calculated as 100% minus reliability.
                </p>
                <p>{dashboard_message}</p>
            </div>
            """,
            unsafe_allow_html=True
        )


# ============================================================
# REGIONAL SNAPSHOT — INTERPRETATION
# ============================================================

st.subheader(
    "Regional Snapshot"
)

# ------------------------------------------------------------
# CREATE SIMPLE INTERPRETATION PHRASES
# ------------------------------------------------------------

if water_access_rate >= 80:

    access_interpretation = (
        "Most of the population has access to water."
    )

elif water_access_rate >= 50:

    access_interpretation = (
        "A majority of the population has access to water, "
        "but a noticeable access gap remains."
    )

else:

    access_interpretation = (
        "Less than half of the population has access to water, "
        "indicating a substantial access gap."
    )


if risk_level.upper() == "HIGH":

    risk_interpretation = (
        "The model indicates a high estimated water-supply risk "
        "for this region."
    )

elif risk_level.upper() == "MEDIUM":

    risk_interpretation = (
        "The model indicates a moderate estimated water-supply risk "
        "for this region."
    )

else:

    risk_interpretation = (
        "The model indicates a relatively low estimated "
        "water-supply risk for this region."
    )


if urban_share > rural_share:

    settlement_interpretation = (
        f"The population is mainly urban, with "
        f"{urban_share:.1f}% living in urban areas."
    )

elif rural_share > urban_share:

    settlement_interpretation = (
        f"The population is mainly rural, with "
        f"{rural_share:.1f}% living in rural areas."
    )

else:

    settlement_interpretation = (
        "The population is almost evenly split "
        "between urban and rural areas."
    )


# ------------------------------------------------------------
# SNAPSHOT CONTENT
# ------------------------------------------------------------

snapshot_items = [

    {
        "number": 1,
        "label": "Water Access",
        "value": (
            f"{water_access_rate:.1f}% have access. "
            f"{access_interpretation}"
        )
    },

    {
        "number": 2,
        "label":" ",
        "value": (
            f"The estimated outage-risk proxy is "
            f"{risk:.1f}%. {risk_interpretation}"
        )
    },

    {
        "number": 3,
        "label": "Population Profile",
        "value": (
            settlement_interpretation
        )
    }
]


render_regional_snapshot_card(
    selected_region,
    snapshot_items
)


import plotly.graph_objects as go


# ============================================================
# 3D-INSPIRED NEON DONUT CHART STYLE
# ============================================================

def style_3d_donut(
    fig,
    colors,
    dark_colors,
    center_value,
    center_label
):
    # Get the existing chart data
    original = fig.data[0]

    labels = list(original.labels)
    values = list(original.values)

    # Dark lower layer creates the illusion of depth
    shadow = go.Pie(
        labels=labels,
        values=values,
        hole=0.60,
        sort=False,
        direction="clockwise",
        rotation=original.rotation,
        marker=dict(
            colors=dark_colors,
            line=dict(
                color="rgba(0,0,0,0)",
                width=3
            )
        ),
        textinfo="none",
        hoverinfo="skip",
        showlegend=False,
        domain=dict(
            x=[0.15, 0.85],
            y=[0.14, 0.78]
        )
    )

    # Style the existing foreground donut
    original.hole = 0.60
    original.sort = False
    original.direction = "clockwise"

    original.domain = dict(
        x=[0.15, 0.85],
        y=[0.18, 0.82]
    )

    original.marker = dict(
    colors=colors,
    line=dict(
        color="rgba(0,0,0,0)",
        width=0
    )
    )

    original.textinfo = "percent"
    original.textposition = "inside"

    original.textfont = dict(
        family="Arial",
        size=15,
        color="white"
    )

    original.hovertemplate = (
        "<b>%{label}</b><br>"
        "Value: %{value:,.0f}<br>"
        "Share: %{percent}"
        "<extra></extra>"
    )

    original.showlegend = True

    # Put the dark layer behind the existing chart
    styled_fig = go.Figure(
        data=[shadow, *fig.data]
    )

    # Keep the dashboard styling consistent
    styled_fig.update_layout(
        height=450,
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        showlegend=True,
        margin=dict(
          l=0,
          r=0,
          t=45,
          b=30
        ),
         title=dict(
            font=dict(
                size=19,
                color="#FFFFFF"
            ),
            x=0.04,
            xanchor="left"
        ),
        legend=dict(
            title_text="",
            orientation="h",
            x=0.5,
            xanchor="center",
            y=-0.08,
            font=dict(
                color="#BAE6FD",
                size=11
            ),
            bgcolor="rgba(0,0,0,0)"
        ),

        annotations=[
            dict(
                text=(
                    f"<b>{center_value}</b>"
                    f"<br>"
                    f"<span style='font-size:10px;"
                    f"color:#67E8F9'>{center_label}</span>"
                ),
                x=0.5,
                y=0.55,
                xref="paper",
                yref="paper",
                showarrow=False,
                align="center",
                font=dict(
                    size=23,
                    color="white",
                    family="Arial"
                )
            )
        ]
    )

    return styled_fig


# ============================================================
# SECTION 2 — REGIONAL CONDITIONS
# ============================================================

section_anchor(
    "conditions-section",
    "REGIONAL CONDITIONS"
)


st.header(
    " Regional Conditions"
)


# ============================================================
# CHART INTERPRETATION BAR
# ============================================================

def chart_interpretation(
    text
):

    safe_text = html_lib.escape(
        str(text)
    )

    interpretation_html = f"""
<div class="chart-interpretation">

    <div class="chart-interpretation-title">
         Interpretation
    </div>

    <div>
        {safe_text}
    </div>

</div>
"""

    st.html(
        interpretation_html
    )

# ============================================================
# ROW 1
# ============================================================

col1, col2 = st.columns(
    2
)


# ------------------------------------------------------------
# POPULATION WATER ACCESS
# ------------------------------------------------------------

with col1:

    with st.container(
        key="pie_population_access",
        border=True
    ):

        access_df = pd.DataFrame(
            {
                "Status": [
                    "With Water Access",
                    "Without Water Access"
                ],

                "Population": [
                    population_with_access,
                    population_without_access
                ]
            }
        )


        access_fig = px.pie(
            access_df,
            names="Status",
            values="Population",
            hole=0.58,
            title="Population Water Access"
        )


        access_fig.update_traces(
            textposition="inside",
            textinfo="percent+label",
            pull=[
                0,
                0.03
            ]
        )


        access_fig.update_layout(

            height=430,

            legend_title_text="",

            paper_bgcolor="#062f49",

            plot_bgcolor="#062f49",

            font=dict(
                color="white"
            ),

            margin=dict(
                l=20,
                r=20,
                t=70,
                b=20
            )
        )


        access_fig = style_3d_donut(
            access_fig,
            colors=[
                "#00E5FF",  # With water access
                "#FF4D6D"   # Without water access
            ],
            dark_colors=[
                "#006B83",
                "#872B42"
            ],
            center_value=f"{water_access_rate:.1f}%",
            center_label="WATER ACCESS"
        )

        st.plotly_chart(
            access_fig,
            use_container_width=True,
            key="population_access_chart"
        )

        if water_access_rate >= 80:

            access_message = (
                f"{water_access_rate:.1f}% of the population "
                f"has access to water. This means water access "
                f"is widespread in {selected_region}, although "
                f"a {water_access_gap:.1f}% access gap remains."
            )

        elif water_access_rate >= 50:

            access_message = (
                f"{water_access_rate:.1f}% of the population "
                f"has access to water. Most residents are served, "
                f"but the {water_access_gap:.1f}% access gap "
                f"remains significant."
            )

        else:

            access_message = (
                f"Only {water_access_rate:.1f}% of the population "
                f"has access to water. The remaining "
                f"{water_access_gap:.1f}% indicates a substantial "
                f"water-access challenge."
            )


        chart_interpretation(
            access_message
        )


# ------------------------------------------------------------
# URBAN VS RURAL
# ------------------------------------------------------------

with col2:

    with st.container(
        key="pie_urban_rural",
        border=True
    ):

        settlement_df = pd.DataFrame(
            {
                "Area": [
                    "Urban Population",
                    "Rural Population"
                ],

                "Population": [
                    urban_population,
                    rural_population
                ]
            }
        )


        settlement_fig = px.pie(
            settlement_df,
            names="Area",
            values="Population",
            hole=0.58,
            title="Urban vs Rural Population"
        )


        settlement_fig.update_traces(
            textposition="inside",
            textinfo="percent+label"
        )


        settlement_fig.update_layout(

            height=430,

            legend_title_text="",

            paper_bgcolor="#062f49",

            plot_bgcolor="#062f49",

            font=dict(
                color="white"
            ),

            margin=dict(
                l=20,
                r=20,
                t=70,
                b=20
            )
        )


        settlement_fig = style_3d_donut(
            settlement_fig,
            colors=[
                "#168BFF",  # Urban
                "#00E5A8"   # Rural
            ],
            dark_colors=[
                "#104C91",
                "#00694F"
            ],
            center_value=f"{urban_share:.1f}%",
            center_label="URBAN SHARE"
        )

        st.plotly_chart(
            settlement_fig,
            use_container_width=True,
            key="urban_rural_chart"
        )

        if urban_share > rural_share:

            settlement_message = (
                f"{selected_region} is predominantly urban. "
                f"Approximately {urban_share:.1f}% of the population "
                f"is urban compared with {rural_share:.1f}% rural."
            )

        elif rural_share > urban_share:

            settlement_message = (
                f"{selected_region} is predominantly rural. "
                f"Approximately {rural_share:.1f}% of the population "
                f"is rural compared with {urban_share:.1f}% urban."
            )

        else:

            settlement_message = (
                f"{selected_region} has an almost even distribution "
                f"between urban and rural populations."
            )


        chart_interpretation(
            settlement_message
        )

# ============================================================
# ROW 2
# ============================================================

col3, col4 = st.columns(
    2
)

# ------------------------------------------------------------
# WATER ACCESS COVERAGE
# ------------------------------------------------------------

with col3:

    with st.container(
        key="pie_access_coverage",
        border=True
    ):

        coverage_df = pd.DataFrame(
            {
                "Measure": [
                    "Water Access Rate",
                    "Water Access Gap"
                ],

                "Percentage": [
                    water_access_rate,
                    water_access_gap
                ]
            }
        )


        coverage_fig = px.pie(
            coverage_df,
            names="Measure",
            values="Percentage",
            hole=0.58,
            title="Water Access Coverage"
        )


        coverage_fig.update_traces(
            textposition="inside",
            textinfo="percent+label"
        )


        coverage_fig.update_layout(

            height=430,

            legend_title_text="",

            paper_bgcolor="#062f49",

            plot_bgcolor="#062f49",

            font=dict(
                color="white"
            ),

            margin=dict(
                l=20,
                r=20,
                t=70,
                b=20
            )
        )

        coverage_fig = style_3d_donut(
            coverage_fig,
            colors=[
                "#00E5FF",  # Access rate
                "#FFB347"   # Access gap
            ],
            dark_colors=[
                "#006B83",
                "#88531F"
            ],
            center_value=f"{water_access_rate:.1f}%",
            center_label="COVERAGE"
        )

        st.plotly_chart(
            coverage_fig,
            use_container_width=True,
            key="water_coverage_chart"
        )


        coverage_message = (
            f"For every 100 percentage points represented in the "
            f"water-access profile, {water_access_rate:.1f} "
            f"represent access while {water_access_gap:.1f} "
            f"represent the remaining access gap."
        )


        chart_interpretation(
            coverage_message
        )


# ------------------------------------------------------------
# RELIABILITY VS OUTAGE RISK
# ------------------------------------------------------------

with col4:

    with st.container(
        key="pie_reliability_risk",
        border=True
    ):

        reliability_df = pd.DataFrame(
            {
                "Measure": [
                    "Predicted Reliability",
                    "Outage Risk Proxy"
                ],

                "Percentage": [
                    reliability,
                    risk
                ]
            }
        )


        reliability_fig = px.pie(
            reliability_df,
            names="Measure",
            values="Percentage",
            hole=0.58,
            title="Reliability & Risk Profile"
        )


        reliability_fig.update_traces(
            textposition="inside",
            textinfo="percent+label"
        )


        reliability_fig.update_layout(

            height=430,

            legend_title_text="",

            paper_bgcolor="#062f49",

            plot_bgcolor="#062f49",

            font=dict(
                color="white"
            ),

            margin=dict(
                l=20,
                r=20,
                t=70,
                b=20
            )
        )


        reliability_fig = style_3d_donut(
              reliability_fig,
              colors=[
                  "#00E5A8",  # Reliability
                  "#FF4D6D"   # Outage risk proxy
              ],
              dark_colors=[
                  "#00694F",
                  "#872B42"
              ],
              center_value=f"{risk:.1f}%",
              center_label="OUTAGE RISK"
          )

        st.plotly_chart(
            reliability_fig,
            use_container_width=True,
            key="reliability_risk_chart"
        )


        if risk_level.upper() == "HIGH":

            reliability_message = (
                f"The model predicts {reliability:.1f}% water-supply "
                f"reliability. The corresponding {risk:.1f}% outage-risk "
                f"proxy places the region in the HIGH risk category."
            )


        elif risk_level.upper() == "MEDIUM":

            reliability_message = (
                f"The model predicts {reliability:.1f}% water-supply "
                f"reliability. The corresponding {risk:.1f}% outage-risk "
                f"proxy indicates a MODERATE level of estimated risk."
            )


        else:

            reliability_message = (
                f"The model predicts {reliability:.1f}% water-supply "
                f"reliability. The corresponding {risk:.1f}% outage-risk "
                f"proxy places the region in the LOW risk category."
            )


        chart_interpretation(
            reliability_message
        )

# ============================================================
# REGIONAL SUMMARY
# ============================================================

st.subheader(
    "Regional Summary"
)


regional_summary_items = [

    {
        "number": 1,
        "label": "Population",
        "value": (
            f"{population:,.0f}"
        )
    },

    {
        "number": 2,
        "label": (
            "Water Access Rate"
        ),
        "value": (
            f"{water_access_rate:.2f}%"
        )
    },

    {
        "number": 3,
        "label": (
            "Water Access Gap"
        ),
        "value": (
            f"{water_access_gap:.2f}%"
        )
    },

    {
        "number": 4,
        "label": (
            "Predicted Reliability"
        ),
        "value": (
            f"{reliability:.2f}%"
        )
    },

    {
        "number": 5,
        "label": (
            "Outage Risk Proxy"
        ),
        "value": (
            f"{risk:.2f}%"
        )
    }
]


render_regional_summary_card(
    selected_region,
    regional_summary_items
)
# ============================================================
# WATER ACCESS LOCATIONS — NAVIGATION TARGET
# ============================================================


st.markdown(
    '<div id="water-access-section"></div>',
    unsafe_allow_html=True,
)

st.subheader("Water Access Locations")

if st.sidebar.button(
    " Water Access Locations",
    key="anlok_goto_water_access",
    use_container_width=True,
):
    st.session_state["anlok_scroll_to_water_access"] = True

# ============================================================
# ANLOK WATER — MUNICIPAL WATER ACCESS LOCATIONS
# Gauteng | OpenStreetMap + Official Municipal Records
# ============================================================

import os
import time
import requests
import pandas as pd
import streamlit as st
import folium

from streamlit_folium import st_folium

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

OFFICIAL_WATER_CSV = (
    "Outputs/Interface/official_water_points.csv"
)

OVERPASS_URLS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
]

# Change this if your app uses a different variable name.
# The selected region must be the municipality name.
municipality = selected_region

# ------------------------------------------------------------
# MUNICIPALITY CENTRE
# ------------------------------------------------------------

@st.cache_data(ttl=86400, show_spinner=False)
def get_municipality_centre(municipality_name):

    url = "https://nominatim.openstreetmap.org/search"

    response = requests.get(
        url,
        params={
            "q": f"{municipality_name}, Gauteng, South Africa",
            "format": "jsonv2",
            "limit": 1,
        },
        headers={
            "User-Agent": "ANLOK-Water-Dashboard/1.0"
        },
        timeout=20,
    )

    response.raise_for_status()
    results = response.json()

    if not results:
        return None

    return [
        float(results[0]["lat"]),
        float(results[0]["lon"]),
    ]


# ------------------------------------------------------------
# OPENSTREETMAP WATER POINTS
# ------------------------------------------------------------

@st.cache_data(ttl=1800, show_spinner=False)
def get_osm_water_points(municipality_name):

    # Search within the named administrative area.
    query = f"""
    [out:json][timeout:60];

    area
      ["boundary"="administrative"]
      ["name"="{municipality_name}"]
      ["admin_level"~"^(4|6|7|8|9|10)$"]
      ->.municipality;

    (
      nwr["amenity"="drinking_water"](area.municipality);
      nwr["drinking_water"="yes"](area.municipality);
      nwr["man_made"="water_tap"](area.municipality);
      nwr["amenity"="water_point"](area.municipality);
      nwr["shop"="water"](area.municipality);
      nwr["vending"="water"](area.municipality);
    );

    out center tags;
    """

    errors = []

    for endpoint in OVERPASS_URLS:
        try:
            response = requests.post(
                endpoint,
                data={"data": query},
                timeout=75,
                headers={
                    "User-Agent": "ANLOK-Water-Dashboard/1.0"
                },
            )

            response.raise_for_status()
            elements = response.json().get("elements", [])

            rows = []

            for element in elements:
                tags = element.get("tags", {})

                # Nodes have direct coordinates; ways/relations
                # usually have a centre in this output format.
                lat = element.get("lat")
                lon = element.get("lon")

                if lat is None or lon is None:
                    centre = element.get("center", {})
                    lat = centre.get("lat")
                    lon = centre.get("lon")

                if lat is None or lon is None:
                    continue

                if tags.get("amenity") == "drinking_water":
                    point_type = "Drinking water point"
                elif tags.get("amenity") == "water_point":
                    point_type = "Water collection point"
                elif tags.get("man_made") == "water_tap":
                    point_type = "Public water tap"
                elif tags.get("shop") == "water":
                    point_type = "Water retailer"
                elif tags.get("vending") == "water":
                    point_type = "Water vending"
                else:
                    point_type = "Water access point (type to verify)"

                rows.append({
                    "name": tags.get(
                        "name",
                        point_type
                    ),
                    "latitude": float(lat),
                    "longitude": float(lon),
                    "location_type": point_type,
                    "municipality": municipality_name,
                    "source": "OpenStreetMap",
                    "verification": "Unverified — OSM record",
                    "operator": tags.get("operator", "Not listed"),
                    "access": tags.get("access", "Not listed"),
                    "opening_hours": tags.get(
                        "opening_hours", "Not listed"
                    ),
                    "drinking_water": tags.get(
                        "drinking_water", "Not specified"
                    ),
                })

            return pd.DataFrame(rows)

        except (requests.RequestException, ValueError) as error:
            errors.append(str(error))
            time.sleep(1)

    raise RuntimeError(
        "OpenStreetMap data could not be retrieved. "
        + " | ".join(errors)
    )


# ------------------------------------------------------------
# OFFICIAL MUNICIPAL RECORDS
# ------------------------------------------------------------

def get_official_water_points(municipality_name):

    columns = [
        "name", "latitude", "longitude",
        "location_type", "municipality", "source",
        "verification", "operator", "access",
        "opening_hours", "drinking_water",
    ]

    if not os.path.exists(OFFICIAL_WATER_CSV):
        return pd.DataFrame(columns=columns)

    data = pd.read_csv(OFFICIAL_WATER_CSV)

    required = {"name", "latitude", "longitude", "municipality"}

    if not required.issubset(data.columns):
        st.warning(
            "The official water-points CSV is missing required "
            "columns: name, latitude, longitude, municipality."
        )
        return pd.DataFrame(columns=columns)

    # Keep records for the selected municipality only.
    data = data[
        data["municipality"].astype(str).str.strip().str.casefold()
        == str(municipality_name).strip().casefold()
    ].copy()

    # A record is verified only if the official dataset explicitly
    # identifies it as verified. Never assume verification.
    if "verified" in data.columns:
        verified = (
            data["verified"].astype(str).str.strip().str.lower()
            .isin(["true", "yes", "1", "verified"])
        )
    else:
        verified = pd.Series(False, index=data.index)

    data["source"] = data.get(
        "source",
        pd.Series("Official municipal record", index=data.index)
    )

    data["verification"] = verified.map({
        True: "Verified — official record",
        False: "Unverified — official record; confirmation needed",
    })

    defaults = {
        "location_type": "Municipal water access point",
        "operator": "Not listed",
        "access": "Not listed",
        "opening_hours": "Not listed",
        "drinking_water": "Not specified",
    }

    for column, default in defaults.items():
        if column not in data.columns:
            data[column] = default

    data["source"] = data["source"].fillna(
        "Official municipal record"
    )

    return data[columns]


# ------------------------------------------------------------
# LOAD AND COMBINE DATA
# ------------------------------------------------------------

st.markdown(
    """
    <div style="
        background:linear-gradient(135deg,#061a2b,#0b3b5a);
        border:1px solid rgba(0,225,255,.35);
        border-radius:18px;
        padding:22px;
        margin:14px 0;
        box-shadow:0 0 22px rgba(0,180,255,.10);
    ">
        <div style="
            color:#67e8f9;
            font-size:12px;
            font-weight:700;
            letter-spacing:2px;
        ">ANLOK WATER • GAUTENG</div>
        <div style="
            color:white;
            font-size:25px;
            font-weight:800;
            margin-top:7px;
        ">Water Access Locations</div>
        <div style="color:#b8d9eb;margin-top:6px;">
            Municipal water access information and source verification
        </div>
    </div>
    """,
    unsafe_allow_html=True,
)

st.caption(f"Selected municipality: {municipality}")

try:
    with st.spinner(
        f"Finding water access locations in {municipality}..."
    ):
        centre = get_municipality_centre(municipality)

        if centre is None:
            st.warning(
                "Could not locate this municipality. "
                "Check its official name."
            )
            st.stop()

        osm_data = get_osm_water_points(municipality)
        official_data = get_official_water_points(municipality)

        # Official records are preferred when coordinates match
        # closely enough to indicate a duplicate.
        # Exact coordinate matches are removed here; near-matches
        # should be reviewed rather than automatically merged.
        if not official_data.empty and not osm_data.empty:
            official_coords = set(
                zip(
                    official_data["latitude"].round(5),
                    official_data["longitude"].round(5),
                )
            )

            osm_data = osm_data[
                ~osm_data.apply(
                    lambda row: (
                        round(row["latitude"], 5),
                        round(row["longitude"], 5),
                    ) in official_coords,
                    axis=1,
                )
            ]

        locations = pd.concat(
            [official_data, osm_data],
            ignore_index=True,
        )

except Exception as error:
    st.error(
        "Water locations could not be loaded. "
        "Check your internet connection or try again later."
    )
    st.caption(str(error))
    st.stop()


# ------------------------------------------------------------
# SUMMARY METRICS
# ------------------------------------------------------------

verified_count = (
    locations["verification"]
    .eq("Verified — official record")
    .sum()
    if not locations.empty else 0
)

osm_count = (
    locations["source"].astype(str)
    .str.contains("OpenStreetMap", case=False)
    .sum()
    if not locations.empty else 0
)

metric1, metric2, metric3 = st.columns(3)

with metric1:
    st.metric("Locations found", len(locations))

with metric2:
    st.metric("Officially verified", int(verified_count))

with metric3:
    st.metric("OpenStreetMap records", int(osm_count))


# ------------------------------------------------------------
# FILTERS
# ------------------------------------------------------------

filter1, filter2 = st.columns(2)

with filter1:
    source_options = ["All sources"]

    if not locations.empty:
        source_options += sorted(
            locations["source"].dropna().astype(str).unique().tolist()
        )

    chosen_source = st.selectbox(
        "Data source",
        source_options,
        key="anlok_water_source_filter",
    )

with filter2:
    verification_options = [
        "All verification statuses",
        "Verified — official record",
        "Unverified — official record; confirmation needed",
        "Unverified — OSM record",
    ]

    chosen_verification = st.selectbox(
        "Verification status",
        verification_options,
        key="anlok_water_verification_filter",
    )

filtered = locations.copy()

if chosen_source != "All sources":
    filtered = filtered[
        filtered["source"].astype(str) == chosen_source
    ]

if chosen_verification != "All verification statuses":
    filtered = filtered[
        filtered["verification"] == chosen_verification
    ]


# ============================================================
# ANLOK WATER — DARK BLUE BASEMAP
# WATER ACCESS LOCATION PINS
# ============================================================

import folium
import streamlit as st
from folium.plugins import BeautifyIcon
from streamlit_folium import st_folium

# ------------------------------------------------------------
# CREATE MAP
# ------------------------------------------------------------

water_map = folium.Map(
    location=centre,
    zoom_start=11,
    tiles=None,
    control_scale=True,
    prefer_canvas=True,
)

folium.TileLayer(
    tiles=(
        "https://basemap.queeniemella.cc/"
        "tiles/countries/{z}/{x}/{y}.png"
    ),
    attr=(
        "Map tiles by Mella | "
        "Map data © OpenStreetMap contributors"
    ),
    name="ANLOK Dark Blue",
    overlay=False,
    control=True,
    max_zoom=20,
    referrer_policy="strict-origin",
).add_to(water_map)


# ------------------------------------------------------------
# ADD ALL WATER ACCESS LOCATION PINS
# ------------------------------------------------------------

pin_count = 0


for _, point in filtered.iterrows():

    # Skip records with missing or invalid coordinates
    try:
        latitude = float(point["latitude"])
        longitude = float(point["longitude"])
    except (ValueError, TypeError, KeyError):
        continue

    if not (-90 <= latitude <= 90 and -180 <= longitude <= 180):
        continue

    if not (latitude == latitude and longitude == longitude):
        continue

    # Determine verification status
    is_verified = (
        point["verification"] == "Verified — official record"
    )

    # Assign marker colour
    marker_colour = (
        "#22c55e" if is_verified
        else "#00e5ff"
        if point["source"] == "OpenStreetMap"
        else "#f59e0b"
    )

    # Build popup
    popup_html = f"""
    <div style="
        font-family:Arial,sans-serif;
        min-width:210px;
        color:#123047;
        line-height:1.6;
    ">
        <h4 style="margin:0 0 8px;color:#075985;">
            {point['municipality']}
        </h4>
        <b>Type:</b> {point['location_type']}<br>
        <b>Source:</b> {point['source']}<br>
        <b>Status:</b> {point['verification']}<br>
        <b>Operator:</b> {point['operator']}<br>
        <b>Access:</b> {point['access']}<br>
        <b>Hours:</b> {point['opening_hours']}<br>
        <b>Drinking water:</b> {point['drinking_water']}<br>
        <b>Coordinates:</b> {latitude:.5f}, {longitude:.5f}
    </div>
    """

    # IMPORTANT: Add the marker INSIDE the loop
    folium.Marker(
        location=[latitude, longitude],
        icon=BeautifyIcon(
            icon="tint",
            icon_shape="marker",
            border_color=marker_colour,
            background_color=marker_colour,
            text_color="#ffffff",
            icon_size=[32, 42],
            icon_anchor=[16, 42],
        ),
        tooltip=str(point["name"]),
        popup=folium.Popup(popup_html, max_width=330),
    ).add_to(water_map)

    pin_count += 1


# ------------------------------------------------------------
# LOCATION COUNT
# ------------------------------------------------------------

st.caption(
    f"Water access locations displayed: {pin_count}"
)


# ------------------------------------------------------------
# MAP LEGEND
# ------------------------------------------------------------

st.markdown(
    """
    <div style="
        color:#b8d9eb;
        font-size:13px;
        margin:8px 0 12px;
        display:flex;
        flex-wrap:wrap;
        gap:16px;
        align-items:center;
    ">
        <span>
            <span style="color:#22c55e;"></span>
            Verified official record
        </span>
        <span>
            <span style="color:#00e5ff;"></span>
            OpenStreetMap record
        </span>
        <span>
            <span style="color:#f59e0b;"></span>
            Other official record awaiting verification
        </span>
    </div>
    """,
    unsafe_allow_html=True,
)


# ------------------------------------------------------------
# DISPLAY INTERACTIVE MAP
# ------------------------------------------------------------

st_folium(
    water_map,
    use_container_width=True,
    height=560,
    key="anlok_municipal_water_access_map",
)


# ------------------------------------------------------------
# LOCATION TABLE
# ------------------------------------------------------------

st.subheader("Location Details")

if filtered.empty:
    st.info(
        "No matching water access locations were found. "
        "This does not mean the municipality has no water points; "
        "the available datasets may be incomplete."
    )
else:
    st.dataframe(
        filtered[
            [
                "name",
                "location_type",
                "source",
                "verification",
                "latitude",
                "longitude",
            ]
        ],
        use_container_width=True,
        hide_index=True,
    )

st.caption(
    "Map data © OpenStreetMap contributors. "
    "Locations are not guaranteed to be operational or safe to drink "
    "from. Verify availability and water quality with the relevant "
    "municipality before visiting."
)


# ============================================================
# SECTION 3 — FEATURE ANALYSIS
# ============================================================

section_anchor(
    "features-section",
    "FEATURE ANALYSIS"
)


st.header(
    " Feature Analysis"
)


# ============================================================
# FEATURE IMPORTANCE — ANLOK 3D-STYLE DARK THEME
# ============================================================

st.subheader("Feature Importance")

try:
    importance_json = api_get("/feature-importance")

    importance = pd.DataFrame(
        importance_json.get("features", [])
    )

    if (
        not importance.empty
        and "feature" in importance.columns
        and "importance_mean" in importance.columns
    ):

        importance["importance_mean"] = pd.to_numeric(
            importance["importance_mean"],
            errors="coerce"
        )

        positive_importance = (
            importance
            .dropna(subset=["importance_mean"])
            .query("importance_mean > 0")
            .sort_values(
                "importance_mean",
                ascending=False
            )
            .head(8)
        )

        if not positive_importance.empty:

            # ------------------------------------------------
            # COLOURS
            # ------------------------------------------------

            feature_colors = [
                "#05716C",
                "#1978A5",
                "#38BDF8",
                "#8B7CF6",
                "#14B8A6",
                "#0E7490",
                "#60A5FA",
                "#64748B"
            ][:len(positive_importance)]

            dark_colors = [
                "#023F3C",
                "#0B3D56",
                "#164C65",
                "#403B78",
                "#07584F",
                "#08465A",
                "#254E80",
                "#374151"
            ][:len(positive_importance)]

            slice_pull = [
                0.025
            ] * len(positive_importance)

# ------------------------------------------------
# CREATE FIGURE
# ------------------------------------------------

            importance_fig = go.Figure()

# LAYER 1 — DARK LOWER DEPTH
            importance_fig.add_trace(
                go.Pie(
                    labels=positive_importance["feature"],
                    values=positive_importance["importance_mean"],
                    hole=0.60,
                    sort=False,
                    direction="clockwise",
                    rotation=90,
                    pull=slice_pull,
                    marker=dict(
                        colors=dark_colors,
                        line=dict(
                            color="#071827",
                            width=1
                        )
                    ),
                    textinfo="none",
                    hoverinfo="skip",
                    showlegend=False,
                    domain=dict(
                        x=[0.04, 0.96],
                        y=[0.07, 0.89]
                    )
                )
            )

# LAYER 2 — MAIN DONUT
            importance_fig.add_trace(
                go.Pie(
                    labels=positive_importance["feature"],
                    values=positive_importance["importance_mean"],
                    hole=0.60,
                    sort=False,
                    direction="clockwise",
                    rotation=90,
                    pull=slice_pull,

                    marker=dict(
                    colors=feature_colors,
                    line=dict(
                    color="#3B91A8",
                    width=1.5
                    )
                    ),
                    textposition="inside",
                    textinfo="percent",
                    textfont=dict(
                        family="Segoe UI, sans-serif",
                        size=12,
                        color="#FFFFFF"
                    ),
                    hovertemplate=(
                        "<b>%{label}</b><br>"
                        "Importance share: %{percent}<br>"
                        "Mean importance: %{value:.5f}"
                        "<extra></extra>"
                    ),
                    showlegend=True,
                    domain=dict(
                        x=[0.04, 0.96],
                        y=[0.10, 0.92]
                    )
                )
            )

# ------------------------------------------------
# DARK BACKGROUND AND LAYOUT
# ------------------------------------------------

            importance_fig.update_layout(
                height=470,
                template="plotly_dark",
                paper_bgcolor="#071827",
                plot_bgcolor="#071827",
                font=dict(
                    family="Segoe UI, sans-serif",
                    color="#E5F3FF",
                    size=12
                ),
                title=dict(
                    text="Top Positive Feature Importance Share",
                    font=dict(
                        size=20,
                        color="#FFFFFF"
                    ),
                    x=0.03,
                    xanchor="left"
                ),
                legend=dict(
                    title_text="",
                    font=dict(
                        color="#D8E8F4",
                        size=12
                    ),
                    bgcolor="rgba(0,0,0,0)"
                ),
                margin=dict(
                    l=15,
                    r=20,
                    t=70,
                    b=15
                )
            )

# ------------------------------------------------
# DISPLAY CHART
# ------------------------------------------------

            st.plotly_chart(
                importance_fig,
                use_container_width=True,
                key="anlok_feature_importance_chart"
            )

        else:
            st.info(
                "No positive feature-importance values "
                "are currently available."
            )

    else:
        st.info(
            "Feature-importance data is not currently available."
        )

except Exception as error:
    st.warning(
        "Feature importance could not be loaded."
    )
    st.write(error)

# ============================================================
# CORRELATION HEATMAP
# ============================================================

st.subheader(
    "Feature Correlation Heatmap"
)


try:

    correlation_json = api_get(
        "/correlations"
    )


    corr_columns = (
        correlation_json.get(
            "columns",
            []
        )
    )


    corr_values = (
        correlation_json.get(
            "matrix",
            []
        )
    )


    if (
        corr_columns
        and
        corr_values
    ):

        corr_matrix = pd.DataFrame(
            corr_values,
            index=corr_columns,
            columns=corr_columns
        )


        correlation_fig = px.imshow(
            corr_matrix,
            x=corr_columns,
            y=corr_columns,
            color_continuous_scale=(
                "RdBu_r"
            ),
            zmin=-1,
            zmax=1,
            text_auto=".2f",
            aspect="auto",
            title=(
                "NB4 Feature "
                "Correlation Heatmap"
            )
        )


        correlation_fig.update_layout(
            height=780,
            paper_bgcolor=(
                "rgba(0,0,0,0)"
            ),
            font=dict(
                color="white"
            )
        )


        st.plotly_chart(
            correlation_fig,
            use_container_width=True
        )


    else:

        st.info(
            "Correlation data "
            "is not available."
        )


except Exception as error:

    st.warning(
        "Correlation heatmap "
        "could not be loaded."
    )

    st.write(
        error
    )


# ============================================================
# REGIONAL PROFILE HEATMAP
# ============================================================

st.subheader(
    "Regional Water Profile Heatmap"
)


try:

    profiles_json = api_get(
        "/profiles"
    )


    profiles = pd.DataFrame(
        profiles_json.get(
            "profiles",
            []
        )
    )


    if not profiles.empty:

        number_option = st.selectbox(
            "Regions to display",
            [
                "Top 10",
                "Top 20",
                "All"
            ],
            key=(
                "heatmap_region_count"
            )
        )


        if (
            number_option
            == "Top 10"
        ):

            profile_view = (
                profiles.head(
                    10
                )
            )


        elif (
            number_option
            == "Top 20"
        ):

            profile_view = (
                profiles.head(
                    20
                )
            )


        else:

            profile_view = (
                profiles.copy()
            )


        heatmap_features = [

            "water_access_rate_pct",

            "water_access_gap_pct",

            "urban_population_share_pct",

            "rural_population_share_pct"
        ]


        available_heatmap_features = [

            feature

            for feature
            in heatmap_features

            if feature
            in profile_view.columns
        ]


        if (
            "wsa_name"
            in profile_view.columns
            and
            available_heatmap_features
        ):

            heatmap_data = (
                profile_view
                .set_index(
                    "wsa_name"
                )[
                    available_heatmap_features
                ]
            )


            profile_heatmap = px.imshow(
                heatmap_data,
                x=(
                    available_heatmap_features
                ),
                y=(
                    heatmap_data.index
                ),
                color_continuous_scale=(
                    "Blues"
                ),
                zmin=0,
                zmax=100,
                aspect="auto",
                title=(
                    "Regional Water "
                    "Profile Heatmap"
                )
            )


            profile_heatmap.update_layout(
                height=max(
                    500,
                    len(
                        heatmap_data
                    ) * 26
                ),
                paper_bgcolor=(
                    "rgba(0,0,0,0)"
                ),
                font=dict(
                    color="white"
                )
            )


            st.plotly_chart(
                profile_heatmap,
                use_container_width=True
            )


        else:

            st.info(
                "The profile data does not contain "
                "the required heatmap fields."
            )


    else:

        st.info(
            "Regional profile data "
            "is not available."
        )


except Exception as error:

    st.warning(
        "Regional profile heatmap "
        "could not be loaded."
    )

    st.write(
        error
    )



# ============================================================
# SECTION 4 — REGIONAL MAP
# ANLOK WATER | GAUTENG MUNICIPAL BOUNDARY RISK MAP
# ============================================================

import os
import json
import re
import pandas as pd
import folium
from streamlit_folium import st_folium

section_anchor("map-section", "REGIONAL MAP")

st.header("South African Water Service Area Map")

st.write(
    "Explore predicted water reliability and outage risk "
    "across the nine Gauteng municipalities."
)

st.subheader("Gauteng Regional Outage Risk Map")

# ============================================================
# 1. FILE PATHS
# ============================================================

BOUNDARY_PATH = os.path.join(
    PROJECT_ROOT,
    "Outputs",
    "Interface",
    "gauteng_municipal_boundaries.geojson"
)

PREDICTIONS_PATH = os.path.join(
    PROJECT_ROOT,
    "Outputs",
    "Predictions",
    "gauteng_municipality_predictions.csv"
)

# ============================================================
# 2. CHECK REQUIRED FILES
# ============================================================

if not os.path.exists(BOUNDARY_PATH):
    st.error(f"Municipal boundary file not found: {BOUNDARY_PATH}")
    st.stop()

if not os.path.exists(PREDICTIONS_PATH):
    st.error(f"Municipality predictions file not found: {PREDICTIONS_PATH}")
    st.stop()

# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def normalize_municipality(name):
    """Standardise municipality names for matching."""
    name = str(name).strip().casefold()
    name = re.sub(r"[^a-z0-9]+", " ", name)

    name = re.sub(
        r"\b(local|metropolitan|municipality|municipalities)\b",
        " ",
        name
    )

    return " ".join(name.split())


def boundary_risk_colour(risk):
    """Assign a colour to a predicted outage risk percentage."""

    if risk is None or pd.isna(risk):
        return "#64748b"

    if risk < 30:
        return "#16a34a"       # Low — green
    elif risk < 60:
        return "#eab308"       # Moderate — yellow
    elif risk < 80:
        return "#f97316"       # High — orange
    else:
        return "#dc2626"       # Very high — red


# ============================================================
# 4. LOAD PREDICTIONS
# ============================================================

try:
    predictions = pd.read_csv(PREDICTIONS_PATH)
except Exception as error:
    st.error(f"Could not read the prediction CSV: {error}")
    st.stop()

required_columns = [
    "wsa_name",
    "predicted_outage_risk_proxy_pct"
]

missing_columns = [
    column for column in required_columns
    if column not in predictions.columns
]

if missing_columns:
    st.error(
        "The prediction CSV is missing these required columns: "
        + ", ".join(missing_columns)
    )
    st.stop()

# Convert risk values to numbers.
predictions["predicted_outage_risk_proxy_pct"] = pd.to_numeric(
    predictions["predicted_outage_risk_proxy_pct"],
    errors="coerce"
)

# Remove records without a municipality name or usable risk value.
predictions = predictions.dropna(
    subset=["wsa_name", "predicted_outage_risk_proxy_pct"]
).copy()

# Use predictions as the source for the map summary and table.
map_df = predictions.copy()

# Standardise the column name used by the summary and table.
map_df["outage_risk_pct"] = map_df[
    "predicted_outage_risk_proxy_pct"
]

# Support alternative reliability column names, if present.
if (
    "predicted_reliability_pct" not in map_df.columns
    and "predicted_reliable_supply_pct" in map_df.columns
):
    map_df["predicted_reliability_pct"] = map_df[
        "predicted_reliable_supply_pct"
    ]

# Create a lookup: normalised municipality name -> risk percentage.
risk_lookup = {}

for _, row in predictions.iterrows():
    municipality_name = row["wsa_name"]
    risk_value = row["predicted_outage_risk_proxy_pct"]

    municipality_key = normalize_municipality(municipality_name)

    if municipality_key:
        risk_lookup[municipality_key] = float(risk_value)


# ============================================================
# 5. LOAD MUNICIPAL BOUNDARIES
# ============================================================

try:
    with open(BOUNDARY_PATH, "r", encoding="utf-8") as file:
        boundary_data = json.load(file)
except Exception as error:
    st.error(f"Could not read the municipal boundary file: {error}")
    st.stop()

matched_features = []
unmatched_boundary_names = []

for feature in boundary_data.get("features", []):
    properties = feature.get("properties", {})

    # MUNICNAME is the municipality-name field in your GeoJSON.
    boundary_name = (
    properties.get("MUNICNAME")
    or properties.get("MUNICIPALITY")
    or properties.get("ADM3_EN")
    or properties.get("NAME_3")
    or properties.get("NAME")
    or properties.get("name")
    )

    if not boundary_name:
        unmatched_boundary_names.append("Unknown boundary name")
        continue

    municipality_key = normalize_municipality(boundary_name)

    if municipality_key not in risk_lookup:
        unmatched_boundary_names.append(boundary_name)
        continue

    risk_value = risk_lookup[municipality_key]

    # Attach the prediction to the polygon for styling and tooltips.
    properties["anlok_region_name"] = str(boundary_name)
    properties["anlok_risk"] = round(risk_value, 1)
    properties["anlok_fill"] = boundary_risk_colour(risk_value)

    matched_features.append(feature)


# ============================================================
# 6. DIAGNOSTICS
# ============================================================

print("Prediction records:", len(predictions))
print(
    "Boundary polygons:",
    len(boundary_data.get("features", []))
)
print("Matched municipal boundaries:", len(matched_features))
print("Prediction municipality names:")
print(predictions["wsa_name"].tolist())
print("Unmatched boundary names:")
print(unmatched_boundary_names)


# ============================================================
# 7. DISPLAY MUNICIPAL BOUNDARY RISK MAP
# ============================================================

st.subheader("Municipal Water Outage Risk Map")

if not matched_features:
    st.error(
        "No municipal boundaries matched the prediction records. "
        "Check the municipality names in the Colab output."
    )

else:
    matched_geojson = {
        "type": "FeatureCollection",
        "features": matched_features
    }

    water_map = folium.Map(
        location=[-26.1, 28.2],
        zoom_start=7,
        tiles="OpenStreetMap",
        control_scale=True
    )

    folium.GeoJson(
        matched_geojson,
        name="Predicted Municipal Outage Risk",

        style_function=lambda feature: {
            "fillColor": feature["properties"]["anlok_fill"],
            "color": "#67e8f9",
            "weight": 2,
            "fillOpacity": 0.70,
        },

        highlight_function=lambda feature: {
            "color": "#ffffff",
            "weight": 4,
            "fillOpacity": 0.90,
        },

        tooltip=folium.GeoJsonTooltip(
            fields=[
                "anlok_region_name",
                "anlok_risk"
            ],
            aliases=[
                "Municipality:",
                "Predicted outage risk proxy (%):"
            ],
            localize=True,
            sticky=True
        )
    ).add_to(water_map)

    folium.LayerControl().add_to(water_map)

    # Zoom the map to the boundaries that were matched.
    water_map.fit_bounds(water_map.get_bounds())

    st_folium(
        water_map,
        use_container_width=True,
        height=620,
        key="anlok_municipal_boundary_risk_map"
    )

    st.caption(
        f"Displaying {len(matched_features)} matched municipal "
        "boundaries. Hover over a municipality to view its predicted "
        "outage risk proxy. Polygon colours represent predicted risk."
    )

    # --------------------------------------------------------
    # RISK COLOUR LEGEND
    # --------------------------------------------------------

    legend_col1, legend_col2, legend_col3, legend_col4 = st.columns(4)

    with legend_col1:
        st.markdown("🟩 **Low**")
        st.caption("Below 30%")

    with legend_col2:
        st.markdown("🟨 **Moderate**")
        st.caption("30% to below 60%")

    with legend_col3:
        st.markdown("🟧 **High**")
        st.caption("60% to below 80%")

    with legend_col4:
        st.markdown("🟥 **Very high**")
        st.caption("80% and above")


# ============================================================
# 8. MAP SUMMARY
# ============================================================

st.subheader("Map Summary")

map_col1, map_col2, map_col3 = st.columns(3)

with map_col1:
    metric_card(
        "Mapped Regions",
        f"{len(matched_features):,}",
        "Municipal boundaries matched"
    )

with map_col2:
    metric_card(
        "Selected Region",
        str(selected_region),
        "Currently selected region"
    )

with map_col3:
    selected_map_row = map_df[
        map_df["wsa_name"].astype(str).str.strip().str.casefold()
        == str(selected_region).strip().casefold()
    ]

    if not selected_map_row.empty:
        selected_risk = selected_map_row.iloc[0][
            "outage_risk_pct"
        ]
    else:
        selected_risk = None

    try:
        selected_risk_value = float(selected_risk)

        if not pd.notna(selected_risk_value):
            selected_risk_value = None

    except (TypeError, ValueError):
        selected_risk_value = None

    if selected_risk_value is None:
        selected_risk_text = "Unavailable"
        selected_risk_level = "Unknown"

    else:
        selected_risk_text = f"{selected_risk_value:.1f}%"

        if selected_risk_value >= 80:
            selected_risk_level = "Very high"
        elif selected_risk_value >= 60:
            selected_risk_level = "High"
        elif selected_risk_value >= 30:
            selected_risk_level = "Moderate"
        else:
            selected_risk_level = "Low"

    metric_card(
        "Selected Risk Proxy",
        selected_risk_text,
        f"{selected_risk_level} risk"
    )


# ============================================================
# 9. REGIONAL PREDICTION TABLE
# ============================================================

with st.expander("View Municipal Water Risk Data"):

    preferred_columns = [
        "wsa_name",
        "population",
        "water_access_rate_pct",
        "predicted_reliability_pct",
        "predicted_outage_risk_proxy_pct",
        "outage_risk_pct",
        "risk_level",
        "latitude",
        "longitude"
    ]

    available_columns = [
        column for column in preferred_columns
        if column in map_df.columns
    ]

    # Avoid showing the same risk percentage twice.
    if (
        "predicted_outage_risk_proxy_pct" in available_columns
        and "outage_risk_pct" in available_columns
    ):
        available_columns.remove("outage_risk_pct")

    if not available_columns:
        st.warning(
            "The prediction CSV does not contain columns "
            "available for the regional table."
        )

    else:
        map_table = map_df[available_columns].copy()

        map_table = map_table.rename(
            columns={
                "wsa_name": "Municipality",
                "population": "Population",
                "water_access_rate_pct": "Water Access (%)",
                "predicted_reliability_pct": "Predicted Reliability (%)",
                "predicted_outage_risk_proxy_pct": "Predicted Outage Risk Proxy (%)",
                "outage_risk_pct": "Outage Risk Proxy (%)",
                "risk_level": "Risk Level",
                "latitude": "Latitude",
                "longitude": "Longitude"
            }
        )

        st.dataframe(
            map_table,
            use_container_width=True,
            hide_index=True
        )

    st.markdown(
        """
        **Predicted Outage Risk Proxy**

        🟢 Low: below 30% &nbsp;&nbsp;
        🟡 Moderate: 30–59.9% &nbsp;&nbsp;
        🟠 High: 60–79.9% &nbsp;&nbsp;
        🔴 Very high: 80–100%
        """
    )



# ============================================================
# JAVASCRIPT SIDEBAR SCROLL + ACTIVE SECTION HIGHLIGHT
# ============================================================

SCROLLSPY_HTML = r"""
<script>

(function() {

    const doc =
        window.parent.document;


    function initAnlokNavigation() {

        const links =
            Array.from(
                doc.querySelectorAll(
                    ".anlok-nav-link"
                )
            );


        if (!links.length) {

            return;
        }


        const sections =
            links
            .map(
                function(link) {

                    const id =
                        link.getAttribute(
                            "data-target"
                        );

                    return (
                        doc.getElementById(
                            id
                        )
                    );
                }
            )
            .filter(Boolean);


        if (!sections.length) {

            return;
        }


        function setActive(
            sectionId
        ) {

            links.forEach(
                function(link) {

                    const target =
                        link.getAttribute(
                            "data-target"
                        );


                    if (
                        target
                        ===
                        sectionId
                    ) {

                        link.classList.add(
                            "active"
                        );

                    }

                    else {

                        link.classList.remove(
                            "active"
                        );
                    }
                }
            );
        }


        /* ================================================
           SIDEBAR CLICK
           ================================================ */

        links.forEach(
            function(link) {

                link.onclick =
                    function(event) {

                        event.preventDefault();


                        const targetId =
                            link.getAttribute(
                                "data-target"
                            );


                        const target =
                            doc.getElementById(
                                targetId
                            );


                        if (target) {

                            target.scrollIntoView(
                                {

                                    behavior:
                                        "smooth",

                                    block:
                                        "start"

                                }
                            );


                            setActive(
                                targetId
                            );
                        }
                    };
            }
        );


        /* ================================================
           REMOVE OLD OBSERVER
           ================================================ */

        if (
            window.parent
            .__anlokScrollObserver
            &&
            typeof
            window.parent
            .__anlokScrollObserver
            .disconnect
            ===
            "function"
        ) {

            window.parent
            .__anlokScrollObserver
            .disconnect();
        }


        /* ================================================
           SCROLL OBSERVER
           ================================================ */

        const observer =
            new window.parent
            .IntersectionObserver(

                function(entries) {

                    const visible =
                        entries
                        .filter(
                            function(entry) {

                                return (
                                    entry
                                    .isIntersecting
                                );
                            }
                        )
                        .sort(
                            function(a, b) {

                                return (
                                    a
                                    .boundingClientRect
                                    .top
                                    -
                                    b
                                    .boundingClientRect
                                    .top
                                );
                            }
                        );


                    if (
                        visible.length
                    ) {

                        setActive(
                            visible[
                                0
                            ]
                            .target
                            .id
                        );
                    }
                },


                {

                    root:
                        null,

                    rootMargin:
                        "-20% 0px -65% 0px",

                    threshold: [

                        0,

                        0.05,

                        0.15,

                        0.30
                    ]
                }
            );


        sections.forEach(
            function(section) {

                observer.observe(
                    section
                );
            }
        );


        window.parent
        .__anlokScrollObserver =
            observer;


        setActive(
            "dashboard-section"
        );
    }


    /*
       Streamlit loads page elements
       progressively, so initialize
       several times.
    */

    setTimeout(
        initAnlokNavigation,
        400
    );


    setTimeout(
        initAnlokNavigation,
        1200
    );


    setTimeout(
        initAnlokNavigation,
        2200
    );


})();

</script>
"""


components.html(
    SCROLLSPY_HTML,
    height=0,
    width=0
)



Overwriting app.py


In [32]:
import json
import os

BOUNDARY_PATH = (
    "/content/ANLOK-WATER-PREDICTION-PROJECT-/"
    "Outputs/Interface/gauteng_municipal_boundaries.geojson"
)

with open(BOUNDARY_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Polygon geometry detail:\n")

for feature in data["features"]:
    props = feature.get("properties", {})
    geometry = feature.get("geometry", {})
    geometry_type = geometry.get("type")
    coordinates = geometry.get("coordinates", [])

    if geometry_type == "Polygon":
        # Count points in the outer boundary ring.
        point_count = len(coordinates[0]) if coordinates else 0

    elif geometry_type == "MultiPolygon":
        point_count = sum(
            len(polygon[0])
            for polygon in coordinates
            if polygon
        )

    else:
        point_count = 0

    name = (
        props.get("MUNICNAME")
        or props.get("MUNICIPALITY")
        or props.get("ADM3_EN")
        or props.get("NAME")
        or props.get("name")
        or "Unknown"
    )

    print(f"{name}: {point_count} boundary points")

Polygon geometry detail:

City of Johannesburg Metropolitan Municipality: 5 boundary points
City of Tshwane Metropolitan Municipality: 5 boundary points
Ekurhuleni Metropolitan Municipality: 5 boundary points
Emfuleni: 5 boundary points
Lesedi: 5 boundary points
Merafong City: 5 boundary points
Midvaal: 5 boundary points
Mogale City: 5 boundary points
Rand West City: 5 boundary points


In [15]:
# ============================================================
# CELL 14 — START STREAMLIT
# ============================================================

!pkill -f "streamlit run app.py" || true

!streamlit run app.py \
    --server.port 8501 \
    --server.address 0.0.0.0 \
    > /content/streamlit.log 2>&1 &


time.sleep(5)

print("Streamlit started.")

print()
print("Streamlit log:")
print("-" * 80)

!cat /content/streamlit.log

^C
Streamlit started.

Streamlit log:
--------------------------------------------------------------------------------


2026-10-17 18:02:11.202 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.86.174.82:8501



In [16]:
!pip install pyngrok
import pyngrok
from pyngrok import ngrok
from getpass import getpass

#3IV8VyMyO9MTsrFXXkSB7f9hmMR_63vrzSUnscGaAbt6Lnqak

ngrok_token = getpass(
    "Enter your ngrok authtoken: "
)

ngrok.set_auth_token(
    ngrok_token
)

# Kill any outstanding ngrok processes
ngrok.kill()

public_url = ngrok.connect(8501)

print(
    "Open your Streamlit dashboard here:"
)

print(public_url)

Enter your ngrok authtoken: ··········
Open your Streamlit dashboard here:
NgrokTunnel: "https://violin-ripening-roast.ngrok-free.dev" -> "http://localhost:8501"
